# LangChain & LangGraph Masterclass — Architecture, Agents, RAG, Multi-Agent & Evaluation

This is a standalone deep-dive on **LangGraph** (and the parts of LangChain you need with it). Where the agentic AI masterclass series taught you *what* agents are and *how* to design them, this notebook teaches you the *tool* — how LangGraph actually works under the hood, so you can build, debug, and customize with complete clarity.

## Who this is for

You already know: how agents work conceptually, how RAG works, the theory of memory systems, the multi-agent patterns (supervisor / swarm / scatter-gather). What you *don't* yet know: what a LangGraph "graph" actually is, what "state" means mechanically, how it handles memory under the hood, and how the execution engine works behind the scenes. This notebook closes exactly that gap.

## What this notebook covers

| Part | Sections | What you'll understand |
|---|---|---|
| **A — Mental model & the engine** | §1-3 | What a graph *is*; the Pregel / BSP execution model; the superstep loop that runs everything |
| **B — State, channels, reducers** | §4-6 | What state really is (channels); how reducers merge updates; the three ways to define state |
| **C — Nodes, edges, control flow** | §7-10 | Nodes as functions; normal / conditional edges; `Command`; `Send` and the map-reduce primitive |
| **D — Building agents** | §11-14 | A ReAct agent from scratch (no abstractions); then `create_agent`; tools; middleware |
| **E — Memory, persistence, HITL, streaming** | §15-19 | Checkpointers; thread IDs; the store (long-term memory); interrupts; the four stream modes |
| **F — RAG & multi-agent in LangGraph** | §20-25 | Naive → agentic RAG (CRAG) as a graph; supervisor; swarm; hierarchical; subgraphs |
| **G — Evaluation & the ecosystem** | §26-30 | LangSmith eval; the framework comparison (LlamaIndex / ADK / OpenAI SDK / Anthropic SDK); customizability; clarity |

## How this notebook is different from the others

The agentic masterclass notebooks (N1-N3) were theory-forward with illustrative code. **This one is code-forward.** Almost every section has runnable LangGraph code. Where the real API needs an LLM key, we use a deterministic fake LLM so everything runs offline — but the *graph structure* is real LangGraph throughout. You can swap the fake LLM for `ChatAnthropic` and it works.

## The "complete flexibility" goal

You asked to understand LangGraph deeply enough for **maximum customizability**. The path to that is understanding the layers:

```
   create_agent          ← highest level, least code, least control
        │  (built on)
   StateGraph (Graph API) ← the level you'll mostly work at
        │  (built on)
   Pregel (runtime)       ← the engine; rarely touched directly, but understanding it = clarity
        │  (built on)
   Channels + Actors      ← the primitives; the BSP model
```

Most tutorials stop at `StateGraph`. We go down to Pregel and channels, because **that's where the "behind the scenes" clarity comes from.** Once you understand that a graph is just actors reading and writing channels in supersteps, every confusing LangGraph behavior becomes obvious.

## Versions (2026)

- **LangGraph v1.0** (stable since October 2025). The API in this notebook is v1.x — `StateGraph`, `add_node`, `add_edge`, `add_conditional_edges`, `compile`, `Command`, `Send`, `interrupt`.
- **LangChain v1.x** — `create_agent` (the successor to the old `AgentExecutor` and `create_react_agent`), middleware system.
- Common deprecated patterns to avoid: `set_entry_point()` (use `add_edge(START, ...)`), `config_schema` (deprecated v0.6, use `context_schema`).

## Setup


In [1]:
# Install LangGraph + LangChain. Most cells run offline with a fake LLM defined below.
# !pip install -q langgraph langchain langchain-core langchain-anthropic langgraph-supervisor langgraph-swarm

# To use a REAL LLM, uncomment and set a key, then swap FakeLLM for ChatAnthropic in any cell:
# import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-sonnet-4-6")

import warnings
warnings.filterwarnings("ignore")

# Check LangGraph is importable and print version
try:
    import langgraph
    print(f"langgraph version: {getattr(langgraph, '__version__', 'installed')}")
except ImportError:
    print("langgraph not installed — run the pip line above. The markdown still teaches the concepts.")

print("Setup done. The fake LLM below lets every graph in this notebook run with no API key.")


langgraph version: installed
Setup done. The fake LLM below lets every graph in this notebook run with no API key.


In [3]:
# A deterministic fake LLM so every graph runs offline. It mimics the LangChain
# chat model interface just enough for our demos: .invoke(messages) -> AIMessage,
# and tool-calling via a simple rule engine.

from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, SystemMessage
from typing import Any
import json
import re

class FakeLLM:
    """Mimics a LangChain chat model. Rule-based, deterministic, no network.
    Swap for ChatAnthropic(model='claude-sonnet-4-6') and the graphs work unchanged."""

    def __init__(self, tools=None, name="fake-llm"):
        self.tools = {t.name: t for t in (tools or [])}
        self.name = name
        self._tool_schemas = tools or []

    def bind_tools(self, tools):
        """LangChain models expose bind_tools; we mirror it."""
        return FakeLLM(tools=tools, name=self.name)

    def invoke(self, messages, **kwargs):
        """Look at the latest human/tool message, decide: call a tool or answer."""
        # Normalize to a list
        if not isinstance(messages, list):
            messages = [messages]

        last_human = None
        has_tool_result = False
        for m in messages:
            role = getattr(m, "type", None) or (m.get("role") if isinstance(m, dict) else None)
            if role in ("human", "user"):
                last_human = m.content if hasattr(m, "content") else m.get("content", "")
            if role in ("tool",):
                has_tool_result = True

        text = (last_human or "").lower()

        # If we already have a tool result in the conversation, synthesize a final answer.
        if has_tool_result:
            # Pull the most recent tool message content
            tool_outputs = [m.content for m in messages
                            if (getattr(m, "type", None) == "tool")]
            return AIMessage(content=f"Based on the tool result ({tool_outputs[-1] if tool_outputs else 'n/a'}), "
                                     f"here is your answer.")

        # Decide whether to call a tool, based on simple keyword rules.
        if "weather" in text and "get_weather" in self.tools:
            return AIMessage(
                content="",
                tool_calls=[{"name": "get_weather", "args": {"city": "Delhi"}, "id": "call_1"}],
            )
        if ("calculate" in text or "multiply" in text or "+" in text or "*" in text) and "calculator" in self.tools:
            return AIMessage(
                content="",
                tool_calls=[{"name": "calculator", "args": {"expression": "21 * 2"}, "id": "call_2"}],
            )
        if "search" in text and "web_search" in self.tools:
            return AIMessage(
                content="",
                tool_calls=[{"name": "web_search", "args": {"query": last_human}, "id": "call_3"}],
            )

        # Default: a plain text answer (no tool call).
        return AIMessage(content=f"[fake-llm] I understood: '{last_human}'. Here is a direct answer.")

# Quick sanity check
_llm = FakeLLM()
print(_llm.invoke([HumanMessage(content="Hello there")]).content)
print("FakeLLM ready. It returns AIMessage objects and supports tool_calls + bind_tools.")


[fake-llm] I understood: 'Hello there'. Here is a direct answer.
FakeLLM ready. It returns AIMessage objects and supports tool_calls + bind_tools.


---
# Part A: Mental Model & the Engine

## §1 — What a "graph" actually is in LangGraph

Forget the word "graph" for a second. The clearest way to understand LangGraph is this:

> **LangGraph is a way to write a program as a set of functions that share one piece of state, where you explicitly declare which function runs after which.**

That's it. The "graph" is just the map of "after function A, run function B (or C, depending on the state)." Three primitives:

| Primitive | What it is | Plain-Python analogy |
|---|---|---|
| **State** | A shared data structure passed to every function | A dict that every function reads and writes |
| **Node** | A function that reads state and returns updates to it | A regular Python function |
| **Edge** | A declaration of what runs next | The control flow (`if/else`, loops) made explicit and external |

### Why not just write normal Python?

You absolutely could write an agent as a `while` loop with `if/else`. People do. So why a graph?

Because three things get painful fast in a plain loop, and LangGraph solves all three for free:

1. **Persistence** — you want to pause the agent (for human approval), save its full state to a database, and resume hours later on a different machine. In a plain `while` loop, the state lives in local variables that vanish when the function returns. In LangGraph, the state is a declared structure that the runtime checkpoints automatically after every step.

2. **Streaming** — you want to emit "the agent just called the weather tool" to the user as it happens. In a plain loop you'd manually thread a callback through everything. In LangGraph, the runtime streams state updates after every node automatically.

3. **Cycles with control** — agents loop (call tool → observe → decide → maybe call another tool). LangChain's older "LCEL" chains were acyclic (a fixed pipeline). LangGraph supports cycles with a recursion limit, conditional routing, and visibility into each iteration.

So the value proposition is: **write your logic as nodes + edges, and get persistence, streaming, and controlled cycles for free.** That's the whole pitch.

### The canonical picture

```
        ┌─────────┐
        │  START  │            ← special entry node
        └────┬────┘
             │  (edge)
             ▼
        ┌─────────┐
        │  node_a │            ← a function: (state) -> state updates
        └────┬────┘
             │  (conditional edge — a function inspects state, returns next node name)
        ┌────┴─────┐
        ▼          ▼
   ┌─────────┐ ┌─────────┐
   │ node_b  │ │ node_c  │
   └────┬────┘ └────┬────┘
        │           │
        └─────┬─────┘
              ▼
        ┌─────────┐
        │   END   │            ← special exit node
        └─────────┘
```

`START` and `END` are special sentinel nodes LangGraph provides. Your real nodes sit in between. Edges (solid) are unconditional; conditional edges (the fork) run a router function to pick the next node.

### LangChain vs LangGraph — the relationship

This confuses everyone at first. The clean split:

| | LangChain | LangGraph |
|---|---|---|
| **What** | Components: LLM wrappers, prompt templates, retrievers, output parsers, tool definitions | An execution engine for stateful, cyclic graphs |
| **Composition** | LCEL — linear, acyclic pipelines (`prompt | llm | parser`) | Graphs — cyclic, stateful, branching |
| **Use for** | A one-shot RAG query, a simple chain | Anything that loops, branches, persists, or has human-in-the-loop |
| **Relationship** | LangGraph nodes often *contain* LangChain components | Built on top of LangChain core |

The mental model: **LangChain gives you the building blocks (an LLM call, a retriever, a tool); LangGraph wires them into a stateful program that can loop and persist.** A LangGraph node is frequently just "call this LangChain LLM with this prompt." They're complementary, not competitors.

> **Interview note.** *"When would you use LangChain vs LangGraph?"* LangChain (LCEL) for linear, one-shot pipelines — a RAG query that retrieves then answers, no loop. LangGraph the moment you need cycles (agent loops), branching (route to different handlers), persistence (resume across sessions), or human-in-the-loop. They compose: LangGraph nodes call LangChain components.


In [5]:
# Your first LangGraph: the smallest possible graph. Three nodes, runs offline.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. STATE — a TypedDict every node reads and writes.
class State(TypedDict):
    value: int
    log: str

# 2. NODES — plain functions. Each takes state, returns a dict of UPDATES (not the whole state).
def node_a(state: State) -> dict:
    print(f"  node_a sees value={state['value']}")
    return {"value": state["value"] + 1, "log": state["log"] + "A"}

def node_b(state: State) -> dict:
    print(f"  node_b sees value={state['value']}")
    return {"value": state["value"] * 10, "log": state["log"] + "B"}

# 3. BUILD the graph
builder = StateGraph(State)
builder.add_node("a", node_a)
builder.add_node("b", node_b)

# 4. EDGES — declare control flow
builder.add_edge(START, "a")   # entry: START -> a
builder.add_edge("a", "b")     # a -> b
builder.add_edge("b", END)     # b -> exit

# 5. COMPILE — turns the builder into an executable graph
graph = builder.compile()

# 6. RUN
print("=== Running the graph ===")
result = graph.invoke({"value": 1, "log": ""})
print(f"\nFinal state: {result}")
print(f"\nNote: each node returned only the fields it changed. The runtime merged them.")
print(f"  value: 1 → (a: +1) → 2 → (b: *10) → 20")
print(f"  log:  '' → 'A' → 'AB'")


=== Running the graph ===
  node_a sees value=1
  node_b sees value=2

Final state: {'value': 20, 'log': 'AB'}

Note: each node returned only the fields it changed. The runtime merged them.
  value: 1 → (a: +1) → 2 → (b: *10) → 20
  log:  '' → 'A' → 'AB'


## §2 — The engine: Pregel and the Bulk Synchronous Parallel model

This is the "behind the scenes" you asked about. **Understanding this one section makes every confusing LangGraph behavior obvious.**

LangGraph's runtime is named **Pregel**, after Google's 2010 paper on large-scale graph processing. Pregel itself is an adaptation of **Leslie Valiant's Bulk Synchronous Parallel (BSP)** model from the 1980s. You don't need the history — you need the execution model, because it explains *everything* about how state updates, parallelism, and checkpointing behave.

### The core idea: actors and channels

In Pregel terms:
- **Actors** = your nodes. They do computation.
- **Channels** = your state fields. Actors read from channels and write to channels.

Actors never talk to each other directly. They **only communicate by reading and writing channels.** Node B doesn't call node A; node A writes to a channel, and node B reads that channel on the next step. This indirection is *why* LangGraph can checkpoint, stream, and parallelize — all communication flows through observable channels.

### Execution happens in supersteps

A LangGraph run is a sequence of **supersteps**. Each superstep has four phases:

```
   ┌─────────────────────────────────────────────────────┐
   │                  ONE SUPERSTEP                       │
   │                                                      │
   │  1. PLAN     Which nodes should run this step?       │
   │              (Nodes whose input channels were        │
   │               updated in the PREVIOUS superstep.)    │
   │                                                      │
   │  2. EXECUTE  Run all selected nodes IN PARALLEL.     │
   │              Each reads the channel state as it was  │
   │              at the START of the superstep.          │
   │              ► Writes are NOT visible to siblings    │
   │                during this superstep.                │
   │                                                      │
   │  3. UPDATE   Apply all node writes to channels,      │
   │              using REDUCERS to merge.                │
   │                                                      │
   │  4. CHECKPOINT  Persist the new channel state to     │
   │                 the checkpointer (if configured).    │
   └─────────────────────────────────────────────────────┘
                            │
                            ▼
              Repeat until no node has new work
              (all "vote to halt" — reach END)
```

### The two consequences that explain everything

#### Consequence 1: within a superstep, nodes don't see each other's writes

If node B and node C run in the *same* superstep (because both were triggered by node A), and B writes `value=5`, **C does not see `value=5`.** C sees the value as it was at the *start* of the superstep. Their writes are merged at the UPDATE phase, *after* both finish.

This is the #1 source of "why didn't my parallel node see the other node's update?" confusion. The answer: **by design — superstep isolation.** Writes become visible only at the next superstep.

#### Consequence 2: parallelism is automatic for nodes triggered together

If node A's edges fan out to B and C (both unconditionally), then B and C are selected in the same PLAN phase and **execute in parallel** in the EXECUTE phase. You don't write any threading code. The BSP model gives you parallelism for free — and the barrier (the UPDATE phase) ensures their results merge deterministically.

This is exactly the "scatter-gather" pattern from the agentic masterclass, but at the framework level: fan out to N nodes, they run in parallel, their writes merge via reducers at the barrier.

### How a node "votes to halt"

A node has no more work when its output channels don't trigger any downstream node. When *all* nodes have halted (execution reaches END with nothing pending), the graph terminates and returns the final channel state. This is why you need a recursion limit — if your edges form a cycle that never halts, the graph would run forever; the limit (default 25) is the safety cutoff.

### Why this matters for you

Three practical payoffs from understanding the engine:

1. **Parallel state bugs make sense now.** "My two parallel nodes both incremented a counter but it only went up by one." Because they both read the same starting value, both wrote `start+1`, and the reducer (default: overwrite) kept one. Fix: use an additive reducer (next section).

2. **You know when things run in parallel.** Nodes triggered in the same superstep run concurrently. Design for it (independent work) or avoid it (sequential edges).

3. **Checkpointing makes sense now.** State is persisted at the end of every superstep, so "resume" means "reload the last superstep's channel state and continue planning." Time travel means "reload an *earlier* superstep's state."

### The two API levels

LangGraph exposes this engine at two levels:

| Level | API | When |
|---|---|---|
| **Graph API** | `StateGraph`, `add_node`, `add_edge` | 95% of the time. Declarative, readable |
| **Functional / Pregel API** | `@entrypoint`, `@task`, raw `Pregel` | When you need imperative control flow or are doing something exotic |

We'll work almost entirely in the Graph API. But knowing the Pregel layer is underneath is what gives you the clarity to debug anything.

> **Interview note.** *"How does LangGraph execute a graph?"* The Pregel / BSP model: execution proceeds in supersteps; each superstep plans which nodes to run (those whose input channels changed), executes them in parallel, then applies their writes to channels via reducers at a synchronization barrier, then checkpoints. Nodes communicate only through channels, never directly. Writes within a superstep are invisible to siblings until the next superstep. That isolation is what enables deterministic parallelism and checkpointing.


In [6]:
# Demonstrating supersteps + parallel execution + the reducer rule.
# IMPORTANT: this cell teaches a real LangGraph behavior that surprises everyone.
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END

# --- The error you WILL hit (explained, not run) ---
# If two parallel nodes both write a field with the DEFAULT reducer, LangGraph raises:
#   InvalidUpdateError: "Can receive only one value per step."
# The default reducer does NOT silently overwrite on concurrent writes — it REJECTS them.
# This is a safety feature: it forces you to declare HOW concurrent writes merge.

# --- The RIGHT version: additive reducer for any field parallel nodes write ---
class State(TypedDict):
    trigger_log: Annotated[list, add]
    additive_counter: Annotated[int, add]   # MUST have a reducer — two nodes write it per step
    results: Annotated[list, add]

def fan_out(state: State) -> dict:
    print("  [superstep 1] fan_out runs (triggers two parallel nodes)")
    return {"trigger_log": ["fan_out"]}

def parallel_node_1(state: State) -> dict:
    print(f"  [superstep 2] node_1 reads additive_counter={state['additive_counter']}")
    return {"additive_counter": 1, "results": ["from_node_1"]}

def parallel_node_2(state: State) -> dict:
    print(f"  [superstep 2] node_2 reads additive_counter={state['additive_counter']}")
    return {"additive_counter": 1, "results": ["from_node_2"]}

builder = StateGraph(State)
builder.add_node("fan_out", fan_out)
builder.add_node("node_1", parallel_node_1)
builder.add_node("node_2", parallel_node_2)
builder.add_edge(START, "fan_out")
builder.add_edge("fan_out", "node_1")   # fan out — both run in the SAME superstep
builder.add_edge("fan_out", "node_2")
builder.add_edge("node_1", END)
builder.add_edge("node_2", END)

graph = builder.compile()

print("=== Both parallel nodes start from additive_counter=0 (sibling isolation) ===\n")
result = graph.invoke({"trigger_log": [], "additive_counter": 0, "results": []})

print(f"\nFinal state: {result}")
print("""
What happened (the key lessons):
  1. Both parallel nodes read additive_counter=0 at the START of superstep 2.
     They could NOT see each other's writes — that's superstep isolation.
  2. Both wrote additive_counter=1. The ADDITIVE reducer summed them: 0 + 1 + 1 = 2.
  3. results accumulated BOTH writes: ['from_node_1', 'from_node_2'].

The critical rule: if MORE THAN ONE node can write a field IN THE SAME SUPERSTEP,
that field MUST have a reducer that merges multiple values (like `add`). Without it,
LangGraph raises InvalidUpdateError — it refuses to guess. That error is a FEATURE:
it stops silent data loss from concurrent writes.
""")


=== Both parallel nodes start from additive_counter=0 (sibling isolation) ===

  [superstep 1] fan_out runs (triggers two parallel nodes)
  [superstep 2] node_1 reads additive_counter=0
  [superstep 2] node_2 reads additive_counter=0

Final state: {'trigger_log': ['fan_out'], 'additive_counter': 2, 'results': ['from_node_1', 'from_node_2']}

What happened (the key lessons):
  1. Both parallel nodes read additive_counter=0 at the START of superstep 2.
     They could NOT see each other's writes — that's superstep isolation.
  2. Both wrote additive_counter=1. The ADDITIVE reducer summed them: 0 + 1 + 1 = 2.
  3. results accumulated BOTH writes: ['from_node_1', 'from_node_2'].

The critical rule: if MORE THAN ONE node can write a field IN THE SAME SUPERSTEP,
that field MUST have a reducer that merges multiple values (like `add`). Without it,
LangGraph raises InvalidUpdateError — it refuses to guess. That error is a FEATURE:
it stops silent data loss from concurrent writes.



## §3 — Reading any LangGraph codebase with confidence

Now that you have the engine model, here's how to read *any* LangGraph code you encounter. Every LangGraph program — no matter how complex — is built from exactly these moves. When you see a confusing codebase, find these six things and the whole thing decodes:

### The six things to find in any LangGraph code

```python
# 1. THE STATE — what data flows through the graph?
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    # ↑ Look at the fields and their reducers. This tells you what the graph tracks.

# 2. THE NODES — what are the units of work?
builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)
# ↑ Each add_node is one function. Read each function to know what it does.

# 3. THE ENTRY — where does it start?
builder.add_edge(START, "agent")
# ↑ Follow START to find the first node.

# 4. THE EDGES — what's the control flow?
builder.add_edge("tools", "agent")                    # unconditional
builder.add_conditional_edges("agent", should_continue) # conditional (router function)
# ↑ Trace the edges to understand the flow. Conditional edges have a router function — read it.

# 5. THE EXITS — when does it stop?
#    Look for edges to END, or conditional edges that can return END.

# 6. THE COMPILE — what's attached?
graph = builder.compile(checkpointer=..., interrupt_before=[...])
# ↑ checkpointer = persistence. interrupt_before/after = human-in-the-loop points.
```

### A reading checklist

When you open an unfamiliar `graph.py`, in order:

1. **Find the State class.** Read its fields and reducers. This is the data model.
2. **Find `compile()`.** Note the checkpointer and any interrupts.
3. **Find `add_edge(START, ...)`.** This is your entry point.
4. **Trace edges forward.** Build the graph in your head (or use `graph.get_graph().draw_mermaid()` to render it).
5. **Read each node function.** Each one takes state, returns updates.
6. **Read each router function** (the second argument to `add_conditional_edges`). These are the decision points.

That's the whole skill. There are no hidden mechanisms — everything is state + nodes + edges + compile config.

### Visualizing a graph (the debugging superpower)

LangGraph can render any compiled graph as a Mermaid diagram. This is the fastest way to understand a graph you didn't write:

```python
# Prints a Mermaid diagram you can paste into any Mermaid viewer
print(graph.get_graph().draw_mermaid())

# Or, in a Jupyter notebook with the right deps, render inline:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())
```

We'll use `draw_mermaid()` (text, no deps) throughout this notebook to make graph structure visible.

### Common things that confuse people, pre-answered

| Confusion | Resolution |
|---|---|
| "Where's the main loop?" | There isn't one in your code — the Pregel runtime is the loop. You declare nodes/edges; it runs supersteps. |
| "Why does my node return a partial dict?" | Nodes return *updates*, not the full state. The runtime merges via reducers. |
| "Why didn't my parallel node see the other's write?" | Superstep isolation. Writes merge at the barrier, visible next superstep. |
| "What's `START` / `END`?" | Sentinel nodes the framework provides. START = entry, END = exit. |
| "Graph not compiled error" | You forgot `.compile()`. The builder isn't executable; the compiled graph is. |
| "Why TypedDict and not a regular class?" | TypedDict is serializable (checkpointing needs that) and supports the reducer-annotation system. |

> **Interview note.** *"Walk me through how you'd understand a LangGraph codebase you've never seen."* Find the State class (the data model), the compile call (persistence + interrupts), the START edge (entry), then trace edges and read node + router functions. Render it with `draw_mermaid()`. Emphasize that there's no hidden control flow — it's all declared in nodes and edges; the Pregel runtime is the loop.


---
# Part B: State, Channels & Reducers

## §4 — State, channels, and the three ways to define state

You now know state is "the data every node reads and writes," and that under the hood each state field is a **channel** in the Pregel model. Let's go deep, because state design is where most LangGraph quality (and most bugs) come from.

### State is your durable contract

Critical mental shift: **the state schema is not just a convenience type — it's what gets serialized to your checkpointer after every superstep.** If you use `PostgresSaver`, every field in your state is written to Postgres on every step. This has consequences:

- **Keep state small.** No giant binary blobs, no 10 MB payloads. Those get written to disk every step.
- **Keep state serializable.** TypedDicts of JSON-compatible types serialize cleanly. Arbitrary Python objects may not.
- **Treat schema changes like database migrations.** Add a field, and old checkpoints don't have it. We covered this failure mode in the agentic masterclass N3 §8.

### The three ways to define state

#### 1. TypedDict (the default, most common)

```python
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str
    step_count: int
```

**Pros**: simple, type-checked by your IDE, serializes cleanly, supports reducer annotations.
**Cons**: no runtime validation (TypedDict is just a hint), no default values.

This is what 90% of LangGraph code uses. Default to it.

#### 2. Pydantic BaseModel (when you want runtime validation)

```python
from pydantic import BaseModel, Field

class AgentState(BaseModel):
    messages: list = Field(default_factory=list)
    user_id: str = ""
    step_count: int = 0
```

**Pros**: runtime validation (bad data raises at the boundary), default values, the full Pydantic ecosystem.
**Cons**: slightly more overhead; reducer support is a bit more involved; serialization needs care.

Use Pydantic when input validation matters — e.g., the state is populated from external/untrusted input.

#### 3. Dataclass (a middle ground)

```python
from dataclasses import dataclass, field

@dataclass
class AgentState:
    messages: list = field(default_factory=list)
    user_id: str = ""
    step_count: int = 0
```

**Pros**: default values, lighter than Pydantic, standard library.
**Cons**: no runtime validation; less common in LangGraph code (you'll see TypedDict far more).

### The recommendation

| Situation | Use |
|---|---|
| Default / learning / most production | **TypedDict** |
| State populated from untrusted external input | **Pydantic** |
| Want defaults without Pydantic weight | **dataclass** |

The agentic masterclass made this point: **Pydantic for tool I/O at API boundaries; TypedDict for LangGraph state.** That holds — the state itself is usually TypedDict; the *tools* the agent calls use Pydantic for their argument schemas.

### Multiple state schemas: input, output, and internal

A subtlety that unlocks cleaner designs: a graph can have **different schemas for input, output, and internal state.**

```python
class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str

class InternalState(TypedDict):   # the full working state
    question: str
    retrieved_docs: list
    answer: str

builder = StateGraph(InternalState, input_schema=InputState, output_schema=OutputState)
```

Now the caller passes only `{question}`, the graph works with the full internal state, and the caller gets back only `{answer}`. The intermediate `retrieved_docs` never leaks out. This is great for keeping a clean public interface over a messy internal process.

### Private state between nodes

You can even have state that only flows between *specific* nodes, not the whole graph — by giving a node a different input/output type. This keeps "scratch" data from polluting the main state. Advanced, but worth knowing it exists when you need it.

### What goes in state vs what doesn't

| Put in state | Keep out of state |
|---|---|
| Conversation messages | Large file contents (write to disk/store, keep a reference) |
| Current plan / todos | Secrets, API keys (use config/context) |
| Tool results you need later | One-off scratch values a single node uses internally |
| Control flags (status, step_count) | The LLM client object (inject via config) |
| Small structured data | Anything > a few hundred KB |

The test: "Does a *later* node need this, and should it survive a checkpoint?" If yes → state. If no → a local variable inside the node.

> **Interview note.** *"How do you decide what goes in LangGraph state?"* State is the durable contract — serialized to the checkpointer every superstep. Put in it only what later nodes need and what should survive a pause/resume: messages, plan, key results, control flags. Keep it small and serializable. Large blobs go to a store with a reference in state; secrets go through config/context, never state.


## §5 — Reducers: how state updates actually merge

A **reducer** is the function that decides how a node's write to a channel combines with the existing value. This is the single most important concept for writing correct LangGraph state.

### The default reducer: overwrite (with a catch)

If you don't annotate a field with a reducer, the default behavior is **overwrite** — the node's write replaces the old value:

```python
class State(TypedDict):
    answer: str    # default reducer: new value replaces old
```

But as we saw in §2's demo, the default reducer **rejects concurrent writes** — if two nodes write `answer` in the same superstep, you get `InvalidUpdateError`. The default reducer assumes exactly one writer per superstep.

### Additive reducers: accumulate

For fields that should accumulate (lists that grow, counters that sum), annotate with a reducer:

```python
from typing import Annotated
from operator import add

class State(TypedDict):
    results: Annotated[list, add]   # writes are concatenated, not replaced
    total: Annotated[int, add]      # writes are summed
```

Now `{"results": ["x"]}` from a node *appends* `"x"` rather than replacing the whole list. And multiple parallel nodes can each append safely.

### The most important reducer: `add_messages`

For conversation history, LangGraph provides a purpose-built reducer: `add_messages`. It's smarter than plain `add`:

```python
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
```

`add_messages` does three things plain `add` doesn't:

1. **Appends new messages** to the list (like `add`).
2. **Deduplicates / updates by ID** — if a message with an existing ID comes in, it *updates* that message rather than appending a duplicate. Critical for streaming, where the same message gets updated as tokens arrive.
3. **Coerces formats** — accepts dicts, tuples, or Message objects and normalizes them to LangChain Message objects.

**Almost every agent uses `Annotated[list, add_messages]` for its `messages` field.** It's the canonical pattern. If you see it, you know "this is conversation history that accumulates."

### Custom reducers: when the built-ins aren't enough

A reducer is just a function `(current_value, new_value) -> merged_value`. You can write your own:

```python
def merge_dicts(current: dict, new: dict) -> dict:
    """Merge new keys into current, rather than replacing the whole dict."""
    return {**current, **new}

class State(TypedDict):
    metadata: Annotated[dict, merge_dicts]
```

Now writing `{"metadata": {"key": "val"}}` merges into the existing metadata dict instead of replacing it. Custom reducers are the escape hatch for any merge logic you need — keep last N items, deduplicate by a field, take the max, etc.

### Reducers and parallelism — the connection

This ties back to the BSP engine (§2). When N parallel nodes write the same channel in one superstep:

- **No reducer (default)** → `InvalidUpdateError`. LangGraph refuses to guess.
- **`add` reducer** → all N writes are reduced together (summed/concatenated).
- **Custom reducer** → your function decides how to merge N writes.

So the reducer is *what makes the scatter-gather pattern work* in LangGraph. Fan out to N nodes, each writes to a list channel with an `add` reducer, and the barrier merges all N results into one list. That's map-reduce, built into the state model.

### A reducer cheat sheet

| Field type | Reducer | Behavior |
|---|---|---|
| A value only one node sets | (none — default) | Overwrite; errors on concurrent write |
| Conversation messages | `add_messages` | Append + dedupe-by-id + format coercion |
| A list that grows | `operator.add` | Concatenate |
| A counter | `operator.add` | Sum |
| A dict that merges | custom `merge_dicts` | Shallow merge |
| Keep last N items | custom | Append then truncate |
| A set | custom (union) | Union |

> **Interview note.** *"What's a reducer in LangGraph and why does `add_messages` exist?"* A reducer defines how a node's write merges with the channel's current value. Default is overwrite (and it rejects concurrent writes). `add_messages` is the special reducer for conversation history — it appends, but also dedupes/updates by message ID (essential for streaming where a message updates as tokens arrive) and coerces dicts/tuples to Message objects. It's the canonical `messages` field annotation.


In [7]:
# Reducers in action: default vs add vs add_messages vs custom.
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage

# --- Custom reducer: keep only the last N items ---
def keep_last_3(current: list, new: list) -> list:
    combined = (current or []) + (new or [])
    return combined[-3:]

# --- Custom reducer: shallow dict merge ---
def merge_dicts(current: dict, new: dict) -> dict:
    return {**(current or {}), **(new or {})}

class State(TypedDict):
    overwrite_field: str                              # default reducer
    accumulate_list: Annotated[list, add]             # additive
    messages: Annotated[list, add_messages]           # message reducer
    recent_events: Annotated[list, keep_last_3]       # custom: bounded list
    metadata: Annotated[dict, merge_dicts]            # custom: dict merge

def step_1(state):
    return {
        "overwrite_field": "first",
        "accumulate_list": ["a"],
        "messages": [HumanMessage(content="Hi")],
        "recent_events": ["e1", "e2"],
        "metadata": {"key1": "val1"},
    }

def step_2(state):
    return {
        "overwrite_field": "second",                  # overwrites "first"
        "accumulate_list": ["b", "c"],                # appends → [a, b, c]
        "messages": [AIMessage(content="Hello!")],    # appends
        "recent_events": ["e3", "e4"],                # [e1,e2,e3,e4] → keep last 3 → [e2,e3,e4]
        "metadata": {"key2": "val2"},                 # merges → {key1, key2}
    }

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", END)
graph = builder.compile()

result = graph.invoke({
    "overwrite_field": "", "accumulate_list": [], "messages": [],
    "recent_events": [], "metadata": {},
})

print("=== After two steps, each field merged by its reducer ===\n")
print(f"  overwrite_field : {result['overwrite_field']!r}")
print(f"    → 'first' then OVERWRITTEN to 'second' (default reducer)\n")
print(f"  accumulate_list : {result['accumulate_list']}")
print(f"    → ['a'] then APPENDED ['b','c'] (add reducer)\n")
print(f"  messages        : {[m.content for m in result['messages']]}")
print(f"    → appended both messages (add_messages reducer)\n")
print(f"  recent_events   : {result['recent_events']}")
print(f"    → [e1,e2] + [e3,e4] = 4 items, KEPT LAST 3 (custom reducer)\n")
print(f"  metadata        : {result['metadata']}")
print(f"    → {{key1}} MERGED with {{key2}} (custom dict-merge reducer)")


=== After two steps, each field merged by its reducer ===

  overwrite_field : 'second'
    → 'first' then OVERWRITTEN to 'second' (default reducer)

  accumulate_list : ['a', 'b', 'c']
    → ['a'] then APPENDED ['b','c'] (add reducer)

  messages        : ['Hi', 'Hello!']
    → appended both messages (add_messages reducer)

  recent_events   : ['e2', 'e3', 'e4']
    → [e1,e2] + [e3,e4] = 4 items, KEPT LAST 3 (custom reducer)

  metadata        : {'key1': 'val1', 'key2': 'val2'}
    → {key1} MERGED with {key2} (custom dict-merge reducer)


## §6 — Nodes in depth: what they can and can't do

A node is a function. But there are details that matter for writing them well.

### The node contract

```python
def my_node(state: State) -> dict:
    # 1. READ from state (don't mutate it in place — return updates instead)
    current = state["some_field"]
    # 2. DO WORK (call an LLM, a tool, compute something)
    result = do_something(current)
    # 3. RETURN UPDATES (a dict of only the fields you changed)
    return {"some_field": result}
```

Three rules:
1. **Read from `state`, don't mutate it.** Return a dict of updates; let the reducer merge.
2. **Return only changed fields.** Returning the whole state works but is wasteful and error-prone.
3. **Return `{}` or `None`** if the node changes nothing (e.g., a pure side-effect node that logs).

### Nodes can be sync or async

```python
def sync_node(state): ...
async def async_node(state): ...     # use when the node does I/O (LLM calls, HTTP, DB)
```

If any node is async, invoke the graph with `await graph.ainvoke(...)`. Async nodes are how you get the concurrency benefits from the agentic masterclass N3 §9 — many graph runs multiplexed on one event loop.

### Nodes receive more than just state: config and runtime

A node can take a second parameter to access **runtime context** — config, the store (long-term memory), and injected dependencies:

```python
from langgraph.runtime import Runtime

def node_with_context(state: State, runtime: Runtime) -> dict:
    # Access config passed at invoke time
    user_id = runtime.context.get("user_id")
    # Access the long-term memory store (if compiled with one)
    store = runtime.store
    return {...}
```

This is how you inject per-request data (user ID, tenant, feature flags) without putting it in state. **Config/context is for request-scoped data that isn't part of the durable state.** (The old `config` parameter still works; `Runtime` is the v1.x way.)

### Node naming and the graph structure

The string you pass to `add_node("name", fn)` is how edges refer to the node. By convention:
- Use clear, verb-ish names: `"call_model"`, `"run_tools"`, `"retrieve"`, `"grade_documents"`.
- The name shows up in traces and the Mermaid diagram, so name for readability.

You can also add a node and let it take the function's name automatically:

```python
builder.add_node(call_model)   # node name becomes "call_model"
```

### Nodes can return `Command` (the modern pattern)

Here's a powerful v1.x capability: a node can return a `Command` object that **both updates state AND decides where to go next** — combining a node and a routing decision in one place:

```python
from langgraph.types import Command
from typing import Literal

def smart_node(state: State) -> Command[Literal["node_b", "node_c"]]:
    if state["value"] > 10:
        return Command(update={"log": "high"}, goto="node_b")
    else:
        return Command(update={"log": "low"}, goto="node_c")
```

This eliminates the need for a separate conditional-edge function in many cases. We cover `Command` fully in §8 — but know that nodes aren't limited to returning plain dicts.

### What a node should NOT do

| Anti-pattern | Why | Instead |
|---|---|---|
| Mutate `state` in place | Breaks the reducer model; unpredictable | Return updates |
| Do unbounded work (infinite loop inside a node) | The runtime can't checkpoint mid-node | Break into multiple nodes |
| Store huge objects in returned state | Serialized every superstep | Write to store/disk, keep a reference |
| Catch-and-swallow all errors silently | Hides failures from the runtime's retry logic | Let errors surface, or return structured error state |
| Call the next node directly | Defeats the graph model | Use edges / Command |

### Node-level configuration: retries and caching

When you add a node, you can attach policies:

```python
from langgraph.types import RetryPolicy, CachePolicy

builder.add_node("flaky_api_call", call_api,
                 retry_policy=RetryPolicy(max_attempts=3))
builder.add_node("expensive_compute", compute,
                 cache_policy=CachePolicy(ttl=300))   # cache results for 5 min
```

`RetryPolicy` gives you the exponential-backoff retry logic from the agentic masterclass N1 §4 — declaratively, per node. `CachePolicy` memoizes a node's output so re-runs with the same input skip the work. Both are node-level config you get for free.

> **Interview note.** *"What can a LangGraph node do beyond returning state updates?"* A node can be sync or async; take a `Runtime` parameter to access config, the long-term store, and injected dependencies; return a `Command` to update state and route in one step; and carry a `RetryPolicy` (declarative backoff) or `CachePolicy` (memoization). The node itself stays a pure-ish function — read state, do work, return updates — but the framework wraps it with retries, caching, and context injection.


---
# Part C: Nodes, Edges & Control Flow

## §7 — Edges: normal, conditional, and the agent loop

Edges declare control flow. There are two kinds, and the agent loop is built from them.

### Normal edges: unconditional transitions

```python
builder.add_edge("a", "b")          # after a, always run b
builder.add_edge(START, "a")        # entry point
builder.add_edge("b", END)          # exit
```

A normal edge means "when the source node finishes, the destination node runs next, no questions asked." You can also fan out (multiple edges from one node → parallel execution) and fan in (multiple edges into one node).

### Conditional edges: the decision mechanism

A conditional edge runs a **router function** that inspects state and returns the name of the next node (or a list of names, or END):

```python
def should_continue(state: State) -> str:
    """Router: inspect state, return the name of the next node."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:        # the LLM wants to call a tool
        return "tools"
    return END                          # the LLM gave a final answer

builder.add_conditional_edges(
    "agent",                            # source node
    should_continue,                    # router function
    {                                   # optional: map router outputs to node names
        "tools": "tools",
        END: END,
    },
)
```

**This is the heart of the agent loop.** After the model node runs, the router checks: did the LLM emit a tool call? If yes → go to the tools node. If no → the LLM gave a final answer → END.

### The canonical ReAct agent loop, drawn

```
   START
     │
     ▼
   ┌───────┐
   │ agent │ ◄──────────────┐    (the model decides)
   └───┬───┘                │
       │                    │
  should_continue?          │
   ┌───┴────┐               │
   ▼        ▼               │
[tools]    END              │
   │                        │
   └────────────────────────┘    (after tools run, go back to the model)
```

Two edges make the loop:
- **Conditional edge** from `agent`: tool call → `tools`, else → END.
- **Normal edge** from `tools` back to `agent`: after running tools, always return to the model.

That cycle — agent → tools → agent → tools → ... → END — is the ReAct loop from the agentic masterclass N1 §1, expressed as a LangGraph. The model keeps looping through tools until it produces a final answer with no tool call.

### Router functions can return lists (fan-out)

A router can return a *list* of node names to trigger multiple nodes in parallel:

```python
def route_to_specialists(state) -> list[str]:
    needed = []
    if state["needs_research"]: needed.append("researcher")
    if state["needs_math"]: needed.append("calculator")
    return needed    # both run in parallel in the next superstep
```

This is the conditional version of fan-out — dynamic parallel dispatch based on state.

### The `path_map` argument (third argument to add_conditional_edges)

The optional dict maps the router's return values to actual node names. Two reasons to use it:

1. **Decoupling** — the router returns abstract labels ("needs_tool"), the map translates to node names. Change the node without changing the router.
2. **Graph validation + visualization** — LangGraph can draw the possible edges if you declare them in the map. Without it, the diagram can't show conditional targets.

```python
builder.add_conditional_edges("agent", should_continue,
                              {"continue": "tools", "end": END})
# router returns "continue" or "end"; map translates to node names
```

### Edges to remember

| Pattern | Code |
|---|---|
| Entry | `add_edge(START, "first")` |
| Exit | `add_edge("last", END)` |
| Sequential | `add_edge("a", "b")` |
| Parallel fan-out | `add_edge("a", "b")` + `add_edge("a", "c")` |
| Conditional | `add_conditional_edges("a", router_fn, path_map)` |
| Dynamic parallel | router returns a `list[str]` |
| Loop | conditional edge back to an earlier node |

> **Interview note.** *"How is an agent loop expressed in LangGraph?"* Two edges: a conditional edge from the model node (router checks for tool calls → route to tools node, else END) and a normal edge from the tools node back to the model node. That cycle is the ReAct loop. The recursion limit (default 25) caps it so a misbehaving loop can't run forever.


## §8 — `Command` and `Send`: the modern control-flow primitives

Two primitives in v1.x give you control flow beyond static edges. Both matter for multi-agent systems.

### `Command`: update state and route in one move

A node can return a `Command` object instead of a plain dict. `Command` combines **a state update** and **a routing decision**:

```python
from langgraph.types import Command
from typing import Literal

def agent_node(state: State) -> Command[Literal["tools", "__end__"]]:
    response = call_model(state)
    if response.tool_calls:
        return Command(
            update={"messages": [response]},   # update state
            goto="tools",                       # AND route to "tools"
        )
    return Command(update={"messages": [response]}, goto END)
```

**Why this matters**: without `Command`, you need a node (returns update) *plus* a separate conditional edge (decides route). With `Command`, the node does both. Cleaner for cases where the routing decision is tightly coupled to what the node just computed.

The `Literal[...]` type hint tells LangGraph (and your IDE) the possible destinations, which enables graph visualization and validation.

### `Command` for multi-agent handoffs

`Command` is *the* mechanism behind multi-agent handoffs (agentic masterclass N1 §17). When one agent hands off to another:

```python
def supervisor(state: State) -> Command[Literal["researcher", "writer", "__end__"]]:
    decision = decide_next_agent(state)
    return Command(
        update={"messages": [decision_message]},
        goto=decision,        # route to the chosen specialist
    )
```

And for handoffs that cross subgraph boundaries (a node in a child graph routing to a node in the parent), `Command` takes a `graph` parameter:

```python
return Command(
    update={"messages": [msg]},
    goto="other_agent",
    graph=Command.PARENT,    # navigate in the PARENT graph, not this subgraph
)
```

This `graph=Command.PARENT` is exactly what `langgraph-swarm`'s handoff tools return under the hood (N1 §14). Now you know what's happening when you saw that in the agentic masterclass.

### `Send`: dynamic map-reduce / fan-out to N instances

`Send` solves a problem static edges can't: **"I don't know until runtime how many parallel branches I need."**

Imagine: the agent retrieved 7 documents and wants to grade each one in parallel. You don't know it's 7 until runtime. `Send` lets a router dispatch N copies of a node, each with its own private input:

```python
from langgraph.types import Send

def dispatch_grading(state: State) -> list[Send]:
    # Create one Send per document — each goes to the "grade" node with that doc
    return [Send("grade_document", {"doc": doc}) for doc in state["retrieved_docs"]]

builder.add_conditional_edges("retrieve", dispatch_grading, ["grade_document"])
```

Each `Send("grade_document", {"doc": doc})` says "run the `grade_document` node with *this* private state." All N runs happen in parallel in the next superstep, and their writes merge via reducers (so `grade_document` should write to a list channel with an `add` reducer).

This is **map-reduce as a first-class primitive**:
- **Map**: `Send` dispatches N parallel node instances.
- **Reduce**: the reducer merges their N outputs.

It's how you build scatter-gather (agentic masterclass N1 §15) in LangGraph when N is dynamic.

### `Command` vs `Send` vs conditional edges — when to use which

| You want to... | Use |
|---|---|
| Route to one of a few known nodes based on state | conditional edge (router fn) |
| Update state AND route, tightly coupled | `Command` |
| Hand off between agents / subgraphs | `Command(goto=..., graph=Command.PARENT)` |
| Fan out to N parallel instances, N known only at runtime | `Send` |
| Static parallel fan-out (fixed N) | multiple normal edges |

### The decision in practice

Most graphs: normal edges + conditional edges. Reach for `Command` when nodes naturally decide their own next step (multi-agent, complex routing). Reach for `Send` when you need dynamic parallelism (grade N docs, process N items, spawn N subagents).

> **Interview note.** *"What's the difference between `Command` and `Send` in LangGraph?"* `Command` lets a node return both a state update and a routing decision (`goto`), and can cross subgraph boundaries with `graph=Command.PARENT` — it's the basis of multi-agent handoffs. `Send` dispatches N parallel instances of a node, each with private input, where N is determined at runtime — it's the map half of map-reduce, with reducers handling the reduce. Conditional edges route to known nodes; `Send` creates a dynamic number of parallel branches.


In [8]:
# Send in action: dynamic map-reduce. Grade N documents in parallel, where N is runtime-determined.
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class State(TypedDict):
    docs: list[str]
    grades: Annotated[list, add]      # parallel grade nodes append here → needs add reducer

def retrieve(state: State) -> dict:
    # Pretend we retrieved a variable number of documents
    docs = ["doc about cats", "doc about dogs", "irrelevant doc about taxes"]
    print(f"  retrieve: got {len(docs)} documents")
    return {"docs": docs}

# The router uses Send to dispatch one grade node PER document, in parallel.
def dispatch_grading(state: State):
    print(f"  dispatch: sending {len(state['docs'])} parallel grade tasks via Send")
    return [Send("grade", {"single_doc": doc}) for doc in state["docs"]]

# The grade node receives a PRIVATE state (just its one doc), not the full state.
def grade(state: dict) -> dict:
    doc = state["single_doc"]
    # Fake grading: relevant unless it mentions taxes
    relevant = "taxes" not in doc
    print(f"    grade: '{doc}' → {'relevant' if relevant else 'irrelevant'}")
    return {"grades": [{"doc": doc, "relevant": relevant}]}

builder = StateGraph(State)
builder.add_node("retrieve", retrieve)
builder.add_node("grade", grade)
builder.add_edge(START, "retrieve")
# Conditional edge using Send for dynamic fan-out
builder.add_conditional_edges("retrieve", dispatch_grading, ["grade"])
builder.add_edge("grade", END)
graph = builder.compile()

print("=== Running dynamic map-reduce with Send ===\n")
result = graph.invoke({"docs": [], "grades": []})

print(f"\nFinal grades (merged from {len(result['grades'])} parallel grade nodes):")
for g in result["grades"]:
    print(f"  {g}")
print(f"""
What happened:
  1. retrieve produced 3 docs (N unknown until runtime).
  2. dispatch_grading returned [Send('grade', {{doc}}) for each doc] → 3 parallel grade nodes.
  3. Each grade node got a PRIVATE state ({{single_doc: ...}}), not the whole state.
  4. All 3 ran in ONE superstep, in parallel.
  5. Their writes to 'grades' merged via the `add` reducer.

This is map-reduce as a first-class LangGraph primitive. Send = map, reducer = reduce.
""")
print(graph.get_graph().draw_mermaid())


=== Running dynamic map-reduce with Send ===

  retrieve: got 3 documents
  dispatch: sending 3 parallel grade tasks via Send
    grade: 'doc about cats' → relevant
    grade: 'doc about dogs' → relevant
    grade: 'irrelevant doc about taxes' → irrelevant

Final grades (merged from 3 parallel grade nodes):
  {'doc': 'doc about cats', 'relevant': True}
  {'doc': 'doc about dogs', 'relevant': True}
  {'doc': 'irrelevant doc about taxes', 'relevant': False}

What happened:
  1. retrieve produced 3 docs (N unknown until runtime).
  2. dispatch_grading returned [Send('grade', {doc}) for each doc] → 3 parallel grade nodes.
  3. Each grade node got a PRIVATE state ({single_doc: ...}), not the whole state.
  4. All 3 ran in ONE superstep, in parallel.
  5. Their writes to 'grades' merged via the `add` reducer.

This is map-reduce as a first-class LangGraph primitive. Send = map, reducer = reduce.

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first


---
# Part D: Building Agents

## §9 — A ReAct agent from scratch (no abstractions)

Time to build a real agent using only the primitives from Parts A-C. **No `create_agent`, no prebuilt anything** — just State, nodes, edges. This is the single most clarifying exercise in LangGraph: once you've built the loop by hand, every prebuilt agent becomes transparent.

### The plan

We're building the canonical ReAct loop:

```
   START → agent → (tool calls?) → tools → agent → ... → END
```

Three pieces:
1. **State** with a `messages` field using `add_messages`.
2. **An `agent` node** that calls the LLM (which may emit tool calls).
3. **A `tools` node** that executes any tool calls and appends results.
4. **A conditional edge** that loops back to `agent` if there were tool calls, else ends.

### The pieces in detail

#### State
```python
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
```
Just conversation history. The `add_messages` reducer appends and dedupes.

#### The agent node
```python
def agent_node(state):
    response = llm_with_tools.invoke(state["messages"])  # may contain tool_calls
    return {"messages": [response]}                       # append the AI's response
```
It calls the LLM with the full message history. The LLM returns an `AIMessage` that either has `tool_calls` (it wants to use a tool) or is a plain answer.

#### The tools node
```python
def tools_node(state):
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        tool = tools_by_name[tc["name"]]
        output = tool.invoke(tc["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}    # append the tool results
```
It reads the last message's tool calls, runs each tool, and appends a `ToolMessage` per call. The `tool_call_id` links each result back to its request — the LLM needs this to match results to calls.

#### The router
```python
def should_continue(state):
    last = state["messages"][-1]
    return "tools" if last.tool_calls else END
```
If the LLM's last message has tool calls → run tools. Otherwise → done.

### The wiring

```python
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")     # ← the loop: after tools, back to agent
graph = builder.compile()
```

That `add_edge("tools", "agent")` is the loop. After tools run, control returns to the agent, which sees the tool results and decides again — call more tools, or give a final answer.

### Why build it by hand?

Because now you understand *exactly* what `create_agent` does. When you later write `create_agent(model, tools)` in one line, you know it's building this graph: an agent node, a tools node, a conditional edge for the loop. No magic. And when you need to customize — add a node between agent and tools, change the routing, inject a guardrail — you know precisely where to cut in.

The code cell below builds and runs this, using our offline FakeLLM with a real tool.


In [9]:
# A complete ReAct agent built from scratch — only State, nodes, edges. Runs offline.
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

# --- 1. Define a real tool ---
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    fake = {"Delhi": "38°C, clear", "Mumbai": "33°C, humid", "Bangalore": "28°C, rainy"}
    return fake.get(city, "Unknown city")

tools = [get_weather]
tools_by_name = {t.name: t for t in tools}

# --- 2. The LLM (our offline FakeLLM, bound to tools). Swap for ChatAnthropic in prod. ---
llm_with_tools = FakeLLM(tools=tools)

# --- 3. State ---
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# --- 4. The agent node ---
def agent_node(state: AgentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    has_calls = bool(getattr(response, "tool_calls", None))
    print(f"  [agent] {'requested tool call' if has_calls else 'gave final answer'}")
    return {"messages": [response]}

# --- 5. The tools node ---
def tools_node(state: AgentState) -> dict:
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        tool_fn = tools_by_name[tc["name"]]
        output = tool_fn.invoke(tc["args"])
        print(f"  [tools] {tc['name']}({tc['args']}) → {output}")
        results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
    return {"messages": results}

# --- 6. The router ---
def should_continue(state: AgentState):
    last = state["messages"][-1]
    if getattr(last, "tool_calls", None):
        return "tools"
    return END

# --- 7. Wire the graph ---
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tools", END])
builder.add_edge("tools", "agent")     # THE LOOP
graph = builder.compile()

# --- 8. Visualize the structure ---
print("=== Graph structure ===")
print(graph.get_graph().draw_mermaid())

# --- 9. Run it ---
print("\n=== Running: 'What is the weather?' ===")
result = graph.invoke({"messages": [HumanMessage(content="What is the weather in Delhi?")]})

print("\n=== Final conversation ===")
for m in result["messages"]:
    role = type(m).__name__.replace("Message", "")
    content = m.content if m.content else f"[tool_calls: {[tc['name'] for tc in m.tool_calls]}]"
    print(f"  {role}: {content}")


=== Graph structure ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


=== Running: 'What is the weather?' ===
  [agent] requested tool call
  [tools] get_weather({'city': 'Delhi'}) → 38°C, clear
  [agent] gave final answer

=== Final conversation ===
  Human: What is the weather in Delhi?
  AI: [tool_calls: ['get_weather']]
  Tool: 38°C, clear
  AI: Based on the tool result (38°C, clear), here is your answer.


## §10 — The prebuilt way: `create_agent`, tools, and middleware

You built the ReAct loop by hand in §9. Now here's the one-liner that does the same thing — and, because you understand what's underneath, you can see exactly what it builds.

### `create_agent` (LangChain v1.x)

```python
from langchain.agents import create_agent

agent = create_agent(
    model="anthropic:claude-sonnet-4-6",   # or a model object
    tools=[get_weather, web_search],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke({"messages": [{"role": "user", "content": "Weather in Delhi?"}]})
```

That's it. Under the hood, `create_agent` builds **exactly the graph you built in §9**: an agent node that calls the model, a tools node (the prebuilt `ToolNode`), a conditional edge for the loop, and the `tools → agent` back-edge. It returns a compiled graph — so everything you know about graphs applies (you can stream it, checkpoint it, inspect it with `draw_mermaid()`).

**Naming note**: `create_agent` is the v1.x name. You'll see older code using `create_react_agent` (from `langgraph.prebuilt`) — same idea, the LangChain v1 `create_agent` is the current entry point. Both build the ReAct graph.

### When to use prebuilt vs build-your-own

| Situation | Use |
|---|---|
| Standard ReAct agent with tools | `create_agent` — don't reinvent it |
| You need a node *between* agent and tools (e.g., a guardrail, a logger) | Build your own, or use middleware |
| Custom routing (not just "tool calls? loop: end") | Build your own graph |
| Multi-agent topology | Build with subgraphs + Command (Part F) |
| Learning / understanding | Build your own once, then use prebuilt |

The rule: **`create_agent` for the common case; drop to the Graph API the moment you need something it doesn't give you.** Because you understand the underlying graph, dropping down is never scary.

### Tools: the `@tool` decorator

A LangChain tool is a function plus a schema. The `@tool` decorator generates the schema from the type hints and docstring:

```python
from langchain_core.tools import tool

@tool
def search_flights(origin: str, destination: str, date: str) -> list[dict]:
    """Search available flights.

    Args:
        origin: IATA airport code, e.g. 'BLR'
        destination: IATA airport code, e.g. 'BOM'
        date: ISO date YYYY-MM-DD
    """
    return [{"flight": "6E-543", "price": 3840}]
```

The docstring becomes the tool description the LLM reads (recall the agentic masterclass N1 §3 — description quality is everything). The type hints become the argument schema. For complex arguments, use a Pydantic model:

```python
from pydantic import BaseModel, Field

class FlightSearch(BaseModel):
    origin: str = Field(description="IATA airport code")
    destination: str = Field(description="IATA airport code")
    date: str = Field(description="ISO date YYYY-MM-DD")

@tool(args_schema=FlightSearch)
def search_flights(origin: str, destination: str, date: str) -> list[dict]:
    """Search available flights."""
    ...
```

### The prebuilt `ToolNode`

LangGraph ships a `ToolNode` that does exactly what your hand-written `tools_node` did — reads tool calls from the last message, runs them, appends `ToolMessage`s — but also handles errors, parallel tool calls, and edge cases:

```python
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([get_weather, search_flights])
builder.add_node("tools", tool_node)
```

Use `ToolNode` instead of hand-writing the tools node unless you need custom tool-execution logic.

### Middleware (LangChain v1.x): the customization layer

Middleware is the v1.x way to inject behavior into `create_agent` without dropping to the raw graph. Middleware can hook **before the model call**, **after the model call**, and **around tool execution**:

```python
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware

agent = create_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[...],
    middleware=[
        SummarizationMiddleware(max_tokens=4000),       # auto-summarize long history
        HumanInTheLoopMiddleware(interrupt_on=["send_email"]),  # pause for approval on sensitive tools
    ],
)
```

Built-in middleware covers the most common needs:
- **`SummarizationMiddleware`** — the summary-buffer pattern from the agentic masterclass N1 §11, automatic.
- **`HumanInTheLoopMiddleware`** — interrupt before specified tools for approval (N1 §9).
- Custom middleware — subclass `AgentMiddleware`, override `before_model` / `after_model` / `wrap_tool_call`.

**This is the customizability story for `create_agent`**: you get the prebuilt loop, and middleware lets you inject cross-cutting behavior (summarization, guardrails, HITL, logging) without rebuilding the graph. When middleware isn't enough, you drop to the Graph API.

### The customizability ladder (full picture)

```
   create_agent (no middleware)        ← simplest; the ReAct loop
        │  add middleware
        ▼
   create_agent + middleware           ← inject summarization, HITL, guardrails
        │  need custom topology
        ▼
   StateGraph (Graph API)              ← build any graph; full control
        │  need imperative style
        ▼
   Functional API (@entrypoint/@task)  ← imperative control flow over the engine
        │  need raw engine access
        ▼
   Pregel (channels + actors)          ← almost never; the bare metal
```

Climb down only as far as you need. Most production agents live at "create_agent + middleware" or "StateGraph."

> **Interview note.** *"When do you use `create_agent` vs building a graph by hand?"* `create_agent` builds the standard ReAct graph (agent node + ToolNode + conditional loop) in one line — use it for the common case. Add middleware (summarization, HITL, guardrails) for cross-cutting concerns. Drop to the `StateGraph` API when you need custom topology — a node between agent and tools, non-standard routing, or multi-agent structure. Because the prebuilt agent IS a graph, dropping down is seamless.


---
# Part E: Memory, Persistence, HITL & Streaming

## §11 — How LangGraph handles memory: checkpointers and the store

You know the theory of agent memory (working / short-term / long-term / episodic). Here's **how LangGraph specifically implements each layer.** This is the part you said you didn't know — so let's be precise.

### LangGraph splits memory into two distinct systems

| Memory type (theory) | LangGraph mechanism | Scope |
|---|---|---|
| **Working memory** | The state itself (the channels) | Current run |
| **Short-term memory** (this conversation) | **Checkpointer** — keyed by `thread_id` | One thread (conversation) |
| **Long-term memory** (across conversations) | **Store** — keyed by namespace + key | Across all threads for a user |
| **Episodic / semantic** | Built on the Store (or external vector DB) | Cross-thread |

The crucial distinction LangGraph makes: **checkpointer = short-term (per-thread), store = long-term (cross-thread).** This trips people up. Let's separate them clearly.

### Short-term memory: the checkpointer

The checkpointer persists the **full graph state** after every superstep, keyed by `thread_id`. This is what makes a conversation continuous.

```python
from langgraph.checkpoint.memory import MemorySaver   # dev
# from langgraph.checkpoint.postgres import PostgresSaver  # production

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Thread A — one conversation
config_a = {"configurable": {"thread_id": "conversation-A"}}
graph.invoke({"messages": [HumanMessage("Hi, I'm Sam")]}, config=config_a)
graph.invoke({"messages": [HumanMessage("What's my name?")]}, config=config_a)
# ↑ Second call REMEMBERS "Sam" because same thread_id loads the prior state.

# Thread B — a DIFFERENT conversation, fresh state
config_b = {"configurable": {"thread_id": "conversation-B"}}
graph.invoke({"messages": [HumanMessage("What's my name?")]}, config=config_b)
# ↑ No memory of Sam — different thread.
```

**The mechanism (read-execute-write cycle)**:
1. **Retrieve** — on invoke with a `thread_id`, load the latest checkpoint for that thread.
2. **Execute** — run the graph, starting from that loaded state.
3. **Persist** — after each superstep, write the new state as a new checkpoint.

The checkpointer keeps a *history* of checkpoints (not just the latest) — which is what enables time travel (§13).

### The checkpointer hierarchy

| Checkpointer | Persistence | Use |
|---|---|---|
| `MemorySaver` / `InMemorySaver` | In-process RAM | Dev, tests, notebooks |
| `SqliteSaver` | Local SQLite file | Local apps, single machine |
| `PostgresSaver` | Postgres | Production |
| Redis checkpointer | Redis | High-throughput production |

**The single most important production fact**: `MemorySaver` loses everything on restart. For production you need `PostgresSaver` (or Redis). The agentic masterclass N3 §8 covered the operational details (TTL, schema versioning, cost).

### Long-term memory: the Store

The checkpointer is *per-thread*. But you want some memory to persist **across** threads — user preferences, facts learned in past conversations. That's the **Store**.

```python
from langgraph.store.memory import InMemoryStore   # dev
# from langgraph.store.postgres import PostgresStore  # production

store = InMemoryStore()
graph = builder.compile(checkpointer=checkpointer, store=store)

# Inside a node, access the store via the runtime:
def node(state, runtime):
    store = runtime.store
    user_id = runtime.context["user_id"]
    # Namespaced by user → cross-thread, per-user memory
    namespace = ("memories", user_id)
    store.put(namespace, "food_preference", {"value": "vegetarian"})
    memories = store.search(namespace)   # retrieve this user's memories
    return {...}
```

The Store is **namespaced** (a tuple like `("memories", user_id)`) and stores key-value pairs. It supports semantic search if you configure it with an embedding model — turning it into the vector-backed long-term memory from the agentic masterclass N1 §5-6.

**The privacy point** from N1 §6 applies here: namespace by `user_id` so retrieval is naturally scoped per user. The store enforces the namespace boundary.

### Putting both together: the complete memory picture

```
   ┌──────────────────────────────────────────────────┐
   │                  One agent run                     │
   │                                                    │
   │   Working memory  =  the State (channels)          │  ← lives during the run
   │                                                    │
   └────────────────────┬───────────────────────────────┘
                        │
          after every superstep
                        │
                        ▼
   ┌──────────────────────────────────────────────────┐
   │   CHECKPOINTER (short-term, per thread_id)         │  ← this conversation
   │   Postgres: full state snapshots, with history     │
   └──────────────────────────────────────────────────┘

   ┌──────────────────────────────────────────────────┐
   │   STORE (long-term, per namespace e.g. user_id)    │  ← across conversations
   │   Postgres + embeddings: facts, preferences        │
   └──────────────────────────────────────────────────┘
```

**The clean mental model**:
- **State** = what the agent is thinking about right now.
- **Checkpointer** = the memory of *this* conversation (resume where you left off).
- **Store** = the memory of *this user* across all conversations (remember their preferences).

That's how LangGraph handles memory, end to end.

> **Interview note.** *"How does LangGraph handle memory?"* Two systems. The **checkpointer** persists full graph state per `thread_id` after every superstep — that's short-term/conversation memory and what enables resume + time travel; `PostgresSaver` in production. The **store** is namespaced key-value (by user_id, say) that persists across threads — that's long-term memory, optionally with embedding-based semantic search. State is working memory; checkpointer is this-conversation memory; store is across-conversation memory.


In [10]:
# Memory in action: checkpointer (short-term, per-thread) + store (long-term, cross-thread).
# Uses the v1.x API: context_schema on StateGraph, context= on invoke, runtime.store in nodes.
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.runtime import Runtime
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]

# Context = request-scoped data (NOT durable state). Passed via context= at invoke.
class Context(TypedDict):
    user_id: str

# A node that uses BOTH the conversation (state) and the long-term store.
def chat_node(state: State, runtime: Runtime[Context]) -> dict:
    last = state["messages"][-1].content
    store = runtime.store                        # long-term store, injected by runtime
    user_id = runtime.context["user_id"]         # request-scoped context
    namespace = ("memories", user_id)            # namespaced per user → privacy boundary

    if "i like" in last.lower() or "i prefer" in last.lower():
        store.put(namespace, "preference", {"text": last})
        reply = "Got it, I'll remember that."
    else:
        memories = list(store.search(namespace))
        if memories:
            recalled = memories[0].value["text"]
            reply = f"I remember you said: '{recalled}'. Responding to '{last}'."
        else:
            reply = f"I have no memories for you yet. You said: '{last}'."
    return {"messages": [AIMessage(content=reply)]}

builder = StateGraph(State, context_schema=Context)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Compile with BOTH a checkpointer (short-term) and a store (long-term)
graph = builder.compile(checkpointer=MemorySaver(), store=InMemoryStore())

# === Demo 1: short-term memory (checkpointer) within one thread ===
print("=== Thread A, turn 1: state a preference ===")
cfg_a = {"configurable": {"thread_id": "thread-A"}}
out = graph.invoke({"messages": [HumanMessage("I like vegetarian food")]},
                   config=cfg_a, context={"user_id": "samarth"})
print(f"  AI: {out['messages'][-1].content}")
print(f"  (state has {len(out['messages'])} messages — checkpointed under thread-A)")

print("\n=== Thread A, turn 2: same thread_id continues the conversation ===")
out = graph.invoke({"messages": [HumanMessage("What should I eat?")]},
                   config=cfg_a, context={"user_id": "samarth"})
print(f"  AI: {out['messages'][-1].content}")
print(f"  (state has {len(out['messages'])} messages — conversation accumulated)")

# === Demo 2: long-term memory (store) across a DIFFERENT thread ===
print("\n=== Thread B (NEW conversation, SAME user): store persists across threads ===")
cfg_b = {"configurable": {"thread_id": "thread-B"}}
out = graph.invoke({"messages": [HumanMessage("Any food tips?")]},
                   config=cfg_b, context={"user_id": "samarth"})
print(f"  AI: {out['messages'][-1].content}")
print(f"  (NEW thread → no conversation history, BUT the store recalled the preference)")

# === Demo 3: different user — store is namespaced, so no leakage ===
print("\n=== Thread C (DIFFERENT user): store namespaced by user_id ===")
cfg_c = {"configurable": {"thread_id": "thread-C"}}
out = graph.invoke({"messages": [HumanMessage("Food tips?")]},
                   config=cfg_c, context={"user_id": "someone_else"})
print(f"  AI: {out['messages'][-1].content}")
print(f"  (different user_id → no access to samarth's memories — privacy by namespace)")

print("""
The two memory systems, made concrete:
  • Checkpointer (thread_id) → Thread A turn 2 saw turn 1. Thread B did NOT. = short-term.
  • Store (namespace=user_id) → Thread B recalled what Thread A saved. = long-term, cross-thread.
  • Namespacing → 'someone_else' couldn't see samarth's memory. = privacy boundary.
""")


=== Thread A, turn 1: state a preference ===
  AI: Got it, I'll remember that.
  (state has 2 messages — checkpointed under thread-A)

=== Thread A, turn 2: same thread_id continues the conversation ===
  AI: I remember you said: 'I like vegetarian food'. Responding to 'What should I eat?'.
  (state has 4 messages — conversation accumulated)

=== Thread B (NEW conversation, SAME user): store persists across threads ===
  AI: I remember you said: 'I like vegetarian food'. Responding to 'Any food tips?'.
  (NEW thread → no conversation history, BUT the store recalled the preference)

=== Thread C (DIFFERENT user): store namespaced by user_id ===
  AI: I have no memories for you yet. You said: 'Food tips?'.
  (different user_id → no access to samarth's memories — privacy by namespace)

The two memory systems, made concrete:
  • Checkpointer (thread_id) → Thread A turn 2 saw turn 1. Thread B did NOT. = short-term.
  • Store (namespace=user_id) → Thread B recalled what Thread A saved. = lon

## §12 — Human-in-the-loop: `interrupt` and resume

You know the HITL theory (agentic masterclass N1 §9). Here's the LangGraph mechanism precisely.

### The `interrupt` primitive

A node calls `interrupt(payload)`. The graph **pauses**, persists its state to the checkpointer, and returns control to the caller with the payload. Later, the caller resumes by invoking the graph with a `Command(resume=value)` — and the `interrupt(...)` call *returns that value*, as if it had been waiting.

```python
from langgraph.types import interrupt, Command

def approval_node(state):
    # Pause here. The payload is what your UI shows the human.
    decision = interrupt({
        "action": state["proposed_action"],
        "question": "Approve this action?",
    })
    # When resumed with Command(resume="approve"), `decision` == "approve"
    if decision == "approve":
        return {"status": "approved"}
    return {"status": "rejected"}
```

### Why this is powerful

The pause is **durable**, not a held thread. When `interrupt` fires:
1. State is checkpointed (so it survives a server restart).
2. `invoke` returns — your server is free, no connection held.
3. Hours or days later, the human clicks approve.
4. You call `graph.invoke(Command(resume="approve"), config={thread_id})`.
5. The graph reloads state, the `interrupt()` call returns `"approve"`, execution continues from exactly there.

This is the agentic masterclass N3 §7 "background + HITL" pattern, built into LangGraph. **`interrupt` requires a checkpointer** — without persistence, there's nothing to resume from.

### Two ways to set up interrupts

#### 1. Dynamic interrupt (inside a node) — the modern way

Call `interrupt()` inside a node, as above. Flexible — you decide at runtime whether to pause, and the payload can include live state.

#### 2. Static interrupt (at compile time)

```python
graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["tools"],     # pause BEFORE the tools node every time
    # interrupt_after=["agent"],    # or pause AFTER a node
)
```

Static interrupts pause at fixed points. Useful for "always review before executing tools." Less flexible than dynamic, but zero code in the node.

### The four HITL patterns (from N1 §9), in LangGraph terms

| Pattern | LangGraph mechanism |
|---|---|
| Approve / reject | `interrupt()` returns the decision; node branches on it |
| Edit the proposed action | `interrupt()` returns the edited value; node uses it |
| Human handoff | `interrupt()` then route to a "human" node |
| Inject missing info | `interrupt()` asking for the field; resume with the value |

### Inspecting the interrupt from outside

After an `invoke` that hits an interrupt, you inspect what it's waiting on:

```python
state = graph.get_state(config)
if state.interrupts:
    # state.interrupts[0].value is the payload you passed to interrupt()
    show_to_user(state.interrupts[0].value)
```

Then resume:

```python
graph.invoke(Command(resume=user_decision), config=config)
```

The code cell demonstrates the full pause-inspect-resume cycle.

> **Interview note.** *"How does human-in-the-loop work in LangGraph?"* A node calls `interrupt(payload)`; the graph checkpoints its state and returns control — no thread held, survives restarts. The UI shows the payload; the human decides; you resume with `Command(resume=value)`, and the `interrupt()` call returns that value, continuing execution from exactly that point. Requires a checkpointer. Static interrupts (`interrupt_before`/`interrupt_after` at compile) pause at fixed nodes; dynamic `interrupt()` pauses conditionally with live payloads.


In [11]:
# Human-in-the-loop: pause at an interrupt, inspect, resume. Runs offline.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

class State(TypedDict):
    proposed_action: str
    status: str

def propose(state: State) -> dict:
    print(f"  [propose] agent wants to: {state['proposed_action']}")
    return {}

def approval_gate(state: State) -> dict:
    # PAUSE. The payload is what your UI would render to the human.
    decision = interrupt({
        "action": state["proposed_action"],
        "question": "Approve this action? (approve/reject)",
    })
    # This line only runs AFTER resume. `decision` is the resume value.
    print(f"  [approval_gate] human decided: {decision!r}")
    return {"status": "approved" if decision == "approve" else "rejected"}

def execute(state: State) -> dict:
    if state["status"] == "approved":
        print(f"  [execute] EXECUTED: {state['proposed_action']}")
        return {"status": "done"}
    print(f"  [execute] SKIPPED (rejected)")
    return {"status": "skipped"}

builder = StateGraph(State)
builder.add_node("propose", propose)
builder.add_node("approval_gate", approval_gate)
builder.add_node("execute", execute)
builder.add_edge(START, "propose")
builder.add_edge("propose", "approval_gate")
builder.add_edge("approval_gate", "execute")
builder.add_edge("execute", END)

# interrupt REQUIRES a checkpointer (state must persist across the pause)
graph = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "approval-1"}}

# === Phase 1: run until the interrupt ===
print("=== Phase 1: agent proposes, graph pauses for approval ===")
result = graph.invoke({"proposed_action": "Send email to 5000 customers", "status": ""},
                      config=config)

# Inspect what it's waiting on (this is what your UI would show)
state = graph.get_state(config)
if state.interrupts:
    print(f"\n  GRAPH PAUSED. Waiting on: {state.interrupts[0].value['question']}")
    print(f"  Proposed action: {state.interrupts[0].value['action']}")
    print(f"  (state is checkpointed — server could restart here and resume later)")

# === Phase 2: human approves; resume ===
print("\n=== Phase 2: human clicks 'approve', graph resumes ===")
final = graph.invoke(Command(resume="approve"), config=config)
print(f"\n  Final status: {final['status']}")

# === Demonstrate rejection on a fresh thread ===
print("\n=== Same flow, but human REJECTS ===")
config2 = {"configurable": {"thread_id": "approval-2"}}
graph.invoke({"proposed_action": "Delete production database", "status": ""}, config=config2)
final = graph.invoke(Command(resume="reject"), config=config2)
print(f"  Final status: {final['status']}")


=== Phase 1: agent proposes, graph pauses for approval ===
  [propose] agent wants to: Send email to 5000 customers

  GRAPH PAUSED. Waiting on: Approve this action? (approve/reject)
  Proposed action: Send email to 5000 customers
  (state is checkpointed — server could restart here and resume later)

=== Phase 2: human clicks 'approve', graph resumes ===
  [approval_gate] human decided: 'approve'
  [execute] EXECUTED: Send email to 5000 customers

  Final status: done

=== Same flow, but human REJECTS ===
  [propose] agent wants to: Delete production database
  [approval_gate] human decided: 'reject'
  [execute] SKIPPED (rejected)
  Final status: skipped


## §13 — Streaming and time travel

Two more capabilities you get for free from the graph model.

### Streaming: four modes

LangGraph can stream a running graph. You call `graph.stream(...)` (sync) or `graph.astream(...)` (async) instead of `invoke`, and iterate over the events. There are four `stream_mode` values:

| `stream_mode` | What it emits | Use for |
|---|---|---|
| `"values"` | The **full state** after each superstep | Watching the whole state evolve |
| `"updates"` | Only the **state diff** each node produced | Progress tracking (which node ran, what it changed) |
| `"messages"` | **LLM tokens** as they stream, plus metadata | Chat UIs — token-by-token output |
| `"custom"` | Whatever you emit via `get_stream_writer()` | Custom progress events from inside nodes |

```python
# Stream state updates (progress)
for chunk in graph.stream({"messages": [...]}, config, stream_mode="updates"):
    print(chunk)   # {"node_name": {state updates from that node}}

# Stream LLM tokens (chat UI)
for token, metadata in graph.stream({"messages": [...]}, config, stream_mode="messages"):
    print(token.content, end="")

# Stream multiple modes at once
for mode, chunk in graph.stream({...}, config, stream_mode=["updates", "messages"]):
    ...
```

This maps directly to the agentic masterclass N3 §7 streaming modes. The `"messages"` mode is what powers token-by-token chat; `"updates"` is what powers "the agent is now searching the web" progress indicators. You can pass a *list* of modes to get several at once.

### Time travel: forking from any past checkpoint

Because the checkpointer keeps a *history* of checkpoints (not just the latest), you can:

1. **List the history** — every state the graph passed through.
2. **Fork from any point** — re-run from an earlier state with different input.

```python
# List every checkpoint for this thread
history = list(graph.get_state_history(config))
for snapshot in history:
    print(snapshot.config["configurable"]["checkpoint_id"], "→ next:", snapshot.next)

# Pick a past checkpoint and fork from it
past = history[2]   # some earlier state
fork_config = past.config   # this config points at that specific checkpoint
# Resume execution FROM that past state — maybe with a different input
graph.invoke(new_input, config=fork_config)
```

**Why time travel matters** (from N3 §8):
- **Debugging** — inspect the exact state at each step to find where things went wrong.
- **What-if analysis** — fork from step 5, try a different path, compare outcomes.
- **Error recovery** — a transient failure at step 7? Re-run from the step-6 checkpoint instead of from scratch.
- **HITL correction** — human edits the state at a past point; re-run from there.

### Updating state manually (the other half of time travel)

You can also *modify* a checkpoint's state directly, then resume:

```python
# Overwrite part of the state at the current checkpoint
graph.update_state(config, {"some_field": "corrected_value"})
# Now resume — the graph continues with the corrected state
graph.invoke(None, config=config)
```

`update_state` respects reducers (so updating a `messages` field appends via `add_messages` unless you specify otherwise). This is how a human "edits the agent's plan" mid-run.

### The connection to everything else

Streaming and time travel both fall out of the same property: **the runtime checkpoints state after every superstep, keeping history.** Streaming = emit each checkpoint as it's made. Time travel = reload an old checkpoint. Both are free consequences of the BSP + checkpointer architecture from §2 and §11. This is the payoff of understanding the engine — these features stop being magic.

> **Interview note.** *"How does streaming work in LangGraph, and what's time travel?"* Streaming: call `graph.stream()` with a `stream_mode` — `"values"` (full state per step), `"updates"` (per-node diffs, for progress), `"messages"` (LLM tokens, for chat UIs), or `"custom"`. Time travel: the checkpointer keeps a history of every superstep's state, so you can list it (`get_state_history`), fork from any past checkpoint, or edit a checkpoint (`update_state`) and resume. Both are consequences of per-superstep checkpointing.


In [12]:
# Streaming modes + time travel, demonstrated offline.
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    counter: int
    log: Annotated[list, add]

def step_1(state): return {"counter": state["counter"] + 1, "log": ["step_1 ran"]}
def step_2(state): return {"counter": state["counter"] + 10, "log": ["step_2 ran"]}
def step_3(state): return {"counter": state["counter"] + 100, "log": ["step_3 ran"]}

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)
graph = builder.compile(checkpointer=MemorySaver())

# === Streaming: mode "updates" shows each node's diff as it runs ===
print("=== stream_mode='updates' (per-node diffs — great for progress UIs) ===")
config = {"configurable": {"thread_id": "stream-1"}}
for chunk in graph.stream({"counter": 0, "log": []}, config, stream_mode="updates"):
    print(f"  {chunk}")

# === Streaming: mode "values" shows full state after each step ===
print("\n=== stream_mode='values' (full state snapshots) ===")
config2 = {"configurable": {"thread_id": "stream-2"}}
for chunk in graph.stream({"counter": 0, "log": []}, config2, stream_mode="values"):
    print(f"  counter={chunk['counter']}, log={chunk['log']}")

# === Time travel: list the checkpoint history ===
print("\n=== Time travel: checkpoint history for thread stream-2 ===")
history = list(graph.get_state_history(config2))
print(f"  {len(history)} checkpoints recorded (newest first):")
for snap in history:
    next_nodes = snap.next if snap.next else "(END)"
    ctr = snap.values.get('counter')
    ctr_str = f"{ctr:>4}" if ctr is not None else "  --"
    print(f"    counter={ctr_str}  → next: {next_nodes}")

# === Time travel: fork from an EARLIER checkpoint ===
print("\n=== Fork from an earlier checkpoint and resume ===")
# history is newest-first; pick a checkpoint from partway through
# Find the checkpoint where step_2 is about to run (counter=1, next=step_2)
fork_point = None
for snap in history:
    if snap.next == ("step_2",):
        fork_point = snap
        break

if fork_point:
    print(f"  Forking from state where counter={fork_point.values['counter']}, next=step_2")
    # Resume from that checkpoint — it will run step_2, step_3 again
    forked = graph.invoke(None, config=fork_point.config)
    print(f"  Re-ran from fork → final counter={forked['counter']}")
    print(f"  (Same as original because deterministic — but you COULD inject different state here)")

# === Manual state edit (the HITL-correction use case) ===
print("\n=== update_state: manually correct state, then resume ===")
if fork_point:
    graph.update_state(fork_point.config, {"counter": 1000})
    edited_state = graph.get_state(fork_point.config)
    print(f"  Overwrote counter to {edited_state.values['counter']} at the fork point")
    resumed = graph.invoke(None, config=fork_point.config)
    print(f"  Resumed with corrected state → final counter={resumed['counter']}")
    print(f"  (1000 + 10 + 100 = {resumed['counter']} — the edit took effect)")


=== stream_mode='updates' (per-node diffs — great for progress UIs) ===
  {'step_1': {'counter': 1, 'log': ['step_1 ran']}}
  {'step_2': {'counter': 11, 'log': ['step_2 ran']}}
  {'step_3': {'counter': 111, 'log': ['step_3 ran']}}

=== stream_mode='values' (full state snapshots) ===
  counter=0, log=[]
  counter=1, log=['step_1 ran']
  counter=11, log=['step_1 ran', 'step_2 ran']
  counter=111, log=['step_1 ran', 'step_2 ran', 'step_3 ran']

=== Time travel: checkpoint history for thread stream-2 ===
  5 checkpoints recorded (newest first):
    counter= 111  → next: (END)
    counter=  11  → next: ('step_3',)
    counter=   1  → next: ('step_2',)
    counter=   0  → next: ('step_1',)
    counter=  --  → next: ('__start__',)

=== Fork from an earlier checkpoint and resume ===
  Forking from state where counter=1, next=step_2
  Re-ran from fork → final counter=111
  (Same as original because deterministic — but you COULD inject different state here)

=== update_state: manually correct st

---
# Part F: RAG & Multi-Agent in LangGraph

## §14 — RAG as a graph: naive → agentic (CRAG)

You know RAG conceptually. Here's how it's expressed in LangGraph — and *why* you'd use a graph for it rather than a simple chain.

### Naive RAG doesn't need a graph

The simplest RAG is a linear pipeline: retrieve → augment prompt → generate. That's an LCEL chain, not a graph:

```python
# This is fine as a plain LangChain chain — no graph needed
chain = retriever | format_docs | prompt | llm | parser
answer = chain.invoke("What is RAG?")
```

**If your RAG is truly linear, use a chain.** The graph earns its place only when you add loops or branches.

### When RAG needs a graph: the agentic patterns

The moment RAG becomes *adaptive* — it decides whether to retrieve, grades what it retrieved, rewrites bad queries, or loops until it has good context — you need cycles and branches. That's a graph. The canonical example is **Corrective RAG (CRAG)**:

```
   START
     │
     ▼
   ┌──────────┐
   │ retrieve │
   └────┬─────┘
        ▼
   ┌──────────────┐
   │ grade_docs   │   ← are the retrieved docs relevant?
   └────┬─────────┘
        │
   relevant?
   ┌────┴─────────────┐
   ▼                  ▼
┌──────────┐   ┌──────────────────┐
│ generate │   │ rewrite_query    │   ← docs were bad; rewrite & retry
└────┬─────┘   └────────┬─────────┘
     ▼                  │
    END                 └──► back to retrieve (the loop)
```

This is exactly the agentic RAG from the RAG masterclass, now as a LangGraph. The structure:
- **retrieve** — pull candidate documents.
- **grade_docs** — an LLM grades each doc for relevance (the "corrective" step).
- **conditional edge** — if docs are good → generate; if bad → rewrite the query and loop back.
- **rewrite_query** — improve the query, then retry retrieval.
- **generate** — produce the answer from good docs.

The loop (rewrite → retrieve → grade) is what a chain *can't* do and a graph *can*.

### The state for a RAG graph

```python
class RAGState(TypedDict):
    question: str
    documents: list           # retrieved docs
    relevant: bool            # did grading pass?
    answer: str
    rewrite_count: int        # cap the loop!
```

Note `rewrite_count` — you cap the corrective loop so a hopeless query doesn't retry forever. This is the recursion-limit discipline applied at the application level.

### Why this maps cleanly to LangGraph

Every agentic-RAG technique from the RAG masterclass becomes a graph shape:

| Technique | Graph shape |
|---|---|
| **Naive RAG** | Linear chain (no graph needed) |
| **Corrective RAG (CRAG)** | retrieve → grade → (good: generate / bad: rewrite → loop) |
| **Self-RAG** | generate → critique → (good: end / bad: regenerate with more retrieval) |
| **Adaptive RAG** | router → (no retrieval / single retrieval / corrective loop) by query complexity |
| **Agentic RAG** | a full agent where "retrieve" is one tool among several |

The last one is key: **agentic RAG is just an agent (Part D) where retrieval is a tool.** The ReAct agent you built decides when to retrieve, can retrieve multiple times, can combine retrieval with other tools. No special "RAG framework" needed — it's the agent loop with a retriever tool.

### The code

The cell below builds a working CRAG graph with the corrective loop, offline. Watch the loop fire: a bad query gets rewritten and retried.

> **Interview note.** *"How would you build corrective RAG in LangGraph?"* State holds question, documents, a relevance flag, and a rewrite counter. Nodes: retrieve, grade_docs (LLM grades relevance), rewrite_query, generate. A conditional edge after grading routes to generate (docs good) or rewrite_query (docs bad), and rewrite loops back to retrieve. The rewrite counter caps the loop. Naive RAG is a linear chain and doesn't need a graph; you reach for the graph exactly when RAG becomes adaptive — grading, rewriting, looping.


In [13]:
# Corrective RAG (CRAG) as a LangGraph, with the corrective loop. Runs offline.
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict):
    question: str
    documents: list
    relevant: bool
    answer: str
    rewrite_count: int

# A tiny fake corpus + retriever
CORPUS = {
    "langgraph": "LangGraph is a library for stateful agent workflows using a graph model.",
    "pregel": "Pregel is the BSP execution engine underneath LangGraph.",
    "reducer": "A reducer defines how node writes merge into a state channel.",
}

def retrieve(state: RAGState) -> dict:
    q = state["question"].lower()
    # Naive keyword retrieval
    docs = [text for key, text in CORPUS.items() if key in q]
    print(f"  [retrieve] query='{state['question']}' → {len(docs)} docs")
    return {"documents": docs}

def grade_docs(state: RAGState) -> dict:
    # Relevant if we found any docs
    relevant = len(state["documents"]) > 0
    print(f"  [grade_docs] relevant={relevant}")
    return {"relevant": relevant}

def rewrite_query(state: RAGState) -> dict:
    # Fake rewrite: map a vague query to a keyword that exists in the corpus
    count = state.get("rewrite_count", 0) + 1
    rewritten = "Tell me about langgraph"   # pretend the LLM improved the query
    print(f"  [rewrite_query] attempt {count}: '{state['question']}' → '{rewritten}'")
    return {"question": rewritten, "rewrite_count": count}

def generate(state: RAGState) -> dict:
    answer = f"Based on {len(state['documents'])} docs: {' '.join(state['documents'])[:120]}"
    print(f"  [generate] producing answer")
    return {"answer": answer}

# The corrective router: good docs → generate; bad docs → rewrite (with a cap)
def decide(state: RAGState) -> str:
    if state["relevant"]:
        return "generate"
    if state.get("rewrite_count", 0) >= 2:    # cap the loop!
        print(f"  [decide] giving up after {state['rewrite_count']} rewrites")
        return "generate"                      # generate with what we have
    return "rewrite_query"

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve)
builder.add_node("grade_docs", grade_docs)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("generate", generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade_docs")
builder.add_conditional_edges("grade_docs", decide, ["generate", "rewrite_query"])
builder.add_edge("rewrite_query", "retrieve")   # THE CORRECTIVE LOOP
builder.add_edge("generate", END)
graph = builder.compile()

print("=== CRAG graph structure ===")
print(graph.get_graph().draw_mermaid())

# Demo 1: a query that retrieves well on the first try
print("\n=== Query 1: 'What is langgraph?' (good first retrieval) ===")
r = graph.invoke({"question": "What is langgraph?", "documents": [], "relevant": False,
                  "answer": "", "rewrite_count": 0})
print(f"  ANSWER: {r['answer']}")

# Demo 2: a vague query that needs the corrective loop
print("\n=== Query 2: 'tell me stuff' (bad retrieval → corrective loop fires) ===")
r = graph.invoke({"question": "tell me stuff", "documents": [], "relevant": False,
                  "answer": "", "rewrite_count": 0})
print(f"  ANSWER: {r['answer']}")
print(f"  (took {r['rewrite_count']} query rewrite(s) — the corrective loop worked)")


=== CRAG graph structure ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_docs(grade_docs)
	rewrite_query(rewrite_query)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_docs -.-> generate;
	grade_docs -.-> rewrite_query;
	retrieve --> grade_docs;
	rewrite_query --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


=== Query 1: 'What is langgraph?' (good first retrieval) ===
  [retrieve] query='What is langgraph?' → 1 docs
  [grade_docs] relevant=True
  [generate] producing answer
  ANSWER: Based on 1 docs: LangGraph is a library for stateful agent workflows using a graph model.

=== Query 2: 'tell me stuff' (bad retrieval → corrective loop fires) ===
  [retrieve] query='tell me stuff' → 0 docs
  [grade_docs] relevant=False
  [rewrite_query] attempt 1: 'tell me stuff' → 'Tell me abou

## §15 — Multi-agent networks in LangGraph: supervisor, swarm, hierarchical, subgraphs

You know the multi-agent patterns from the agentic masterclass (N1 §12-17). Here's how each is *built* in LangGraph — the structures behind `langgraph-supervisor` and `langgraph-swarm`, plus the subgraph mechanism that makes hierarchical systems possible.

### The foundational idea: agents are nodes (or subgraphs)

In LangGraph multi-agent systems, each "agent" is either:
- **A node** — a function that runs that agent's logic, or
- **A subgraph** — an entire compiled graph embedded as a node in a parent graph.

Multi-agent coordination is then just **routing between these nodes/subgraphs** — using the conditional edges, `Command`, and `Send` primitives you already learned. There's no separate "multi-agent engine"; it's the same graph model, composed.

### Pattern 1: Supervisor

A central supervisor node routes to specialist nodes; after each specialist, control returns to the supervisor.

```
        ┌────────────┐
        │ supervisor │ ◄──────────┐
        └─────┬──────┘            │
        (routes via Command/      │
         conditional edge)        │
       ┌──────┼──────┐            │
       ▼      ▼      ▼            │
  [research][code][write]         │
       │      │      │            │
       └──────┴──────┴────────────┘
            (each returns to supervisor)
```

**How it's built**: the supervisor is a node that returns `Command(goto=specialist_name)`. Each specialist is a node with an edge back to the supervisor. The supervisor's logic (an LLM call, usually) decides which specialist to route to next, or to END.

`langgraph-supervisor`'s `create_supervisor(...)` builds exactly this graph for you — but now you know it's a supervisor node + specialist nodes + routing edges. You could build it by hand with `Command`.

### Pattern 2: Swarm

No central supervisor — agents hand off to each other directly via handoff tools.

```
   [agent_A] ──handoff──► [agent_B] ──handoff──► [agent_C]
        ▲                                            │
        └────────────────handoff─────────────────────┘
```

**How it's built**: each agent is a node. A "handoff tool" is a tool that, when called, returns `Command(goto="other_agent")`. Whichever agent is active handles the turn; calling a handoff tool routes to a peer. The "active agent" is tracked in state.

`langgraph-swarm`'s `create_swarm(...)` builds this. The handoff tools return `Command(goto=..., graph=Command.PARENT)` — exactly the `Command` mechanism from §8.

### Pattern 3: Hierarchical (subgraphs)

The key enabling mechanism: **a compiled graph can be a node inside another graph.** This is how you build supervisors-of-supervisors.

```python
# A team is its own graph
research_team = build_research_team().compile()

# The top-level graph uses the team graph AS A NODE
top_builder = StateGraph(TopState)
top_builder.add_node("research_team", research_team)   # ← a whole graph as one node
top_builder.add_node("writing_team", writing_team)
top_builder.add_node("top_supervisor", top_supervisor)
```

When the parent graph routes to `"research_team"`, that entire subgraph runs (its own supervisor, its own specialists), then returns control to the parent. This composes infinitely — teams of teams.

**The state-sharing rule for subgraphs**: if the subgraph's state schema shares keys with the parent (like `messages`), those keys flow through automatically. If the schemas differ, you transform state at the boundary (a wrapper node maps parent state → subgraph state and back).

### Pattern 4: Scatter-gather (with Send)

The map-reduce pattern from §8, applied to agents: a coordinator uses `Send` to spawn N subagent invocations in parallel, and a reducer merges their outputs.

```python
def dispatch(state):
    return [Send("subagent", {"task": t}) for t in state["subtasks"]]
```

This is the Anthropic research-system pattern (N1 §15) — lead spawns parallel subagents — built with `Send`.

### Choosing the structure (recap from N1, now with the mechanism)

| Pattern | LangGraph mechanism | When |
|---|---|---|
| Supervisor | supervisor node + `Command`/conditional routing | 3-7 specialists, central control |
| Swarm | handoff tools returning `Command(goto=...)` | peer-to-peer, lower latency |
| Hierarchical | subgraphs as nodes | 10+ specialists in teams |
| Scatter-gather | `Send` + reducer | dynamic parallel subtasks |

### The big realization

There is no "multi-agent framework" hiding inside LangGraph. **Multi-agent systems are just graphs composed of agent-nodes and agent-subgraphs, routed with the primitives you already know** (conditional edges, `Command`, `Send`, subgraphs). The `langgraph-supervisor` and `langgraph-swarm` packages are thin conveniences over these primitives. Once you see that, you can build *any* topology — including ones no library provides — because you're working at the primitive level.

The code cell builds a supervisor multi-agent system by hand with `Command`, so you can see there's no magic.

> **Interview note.** *"How are multi-agent systems built in LangGraph?"* Each agent is a node or a subgraph; coordination is routing between them. Supervisor = a central node returning `Command(goto=specialist)` with specialists edging back. Swarm = handoff tools that return `Command(goto=peer, graph=Command.PARENT)`. Hierarchical = compiled subgraphs used as nodes in a parent graph. Scatter-gather = `Send` to spawn N parallel subagents + a reducer to merge. The `langgraph-supervisor`/`-swarm` libraries are conveniences over these primitives — you can build any topology by hand.


In [14]:
# A supervisor multi-agent system built BY HAND with Command — no library. Runs offline.
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

class MultiAgentState(TypedDict):
    task: str
    findings: Annotated[list, add]
    completed_agents: Annotated[list, add]
    final_report: str

# --- The supervisor: decides which specialist runs next (returns a Command) ---
def supervisor(state: MultiAgentState) -> Command[Literal["researcher", "writer", "__end__"]]:
    completed = state.get("completed_agents", [])
    if "researcher" not in completed:
        print("  [supervisor] → routing to researcher")
        return Command(goto="researcher")
    if "writer" not in completed:
        print("  [supervisor] → routing to writer")
        return Command(goto="writer")
    print("  [supervisor] → all done, ending")
    return Command(goto=END)

# --- Specialist nodes: each does work, records completion, returns to supervisor ---
def researcher(state: MultiAgentState) -> Command[Literal["supervisor"]]:
    finding = f"Research on '{state['task']}': LangGraph uses a BSP/Pregel engine."
    print(f"  [researcher] {finding}")
    return Command(
        update={"findings": [finding], "completed_agents": ["researcher"]},
        goto="supervisor",     # hand back to supervisor
    )

def writer(state: MultiAgentState) -> Command[Literal["supervisor"]]:
    report = f"REPORT: {' '.join(state['findings'])}"
    print(f"  [writer] composed report from {len(state['findings'])} finding(s)")
    return Command(
        update={"final_report": report, "completed_agents": ["writer"]},
        goto="supervisor",
    )

builder = StateGraph(MultiAgentState)
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)
builder.add_edge(START, "supervisor")
# Note: NO explicit edges from specialists — Command(goto=...) handles routing!
graph = builder.compile()

print("=== Supervisor multi-agent graph (built by hand with Command) ===")
print(graph.get_graph().draw_mermaid())

print("\n=== Running ===")
result = graph.invoke({
    "task": "Explain the LangGraph engine",
    "findings": [], "completed_agents": [], "final_report": "",
})
print(f"\nFinal report: {result['final_report']}")
print(f"Agents that ran: {result['completed_agents']}")
print(f"""
Key point: there are NO explicit edges between supervisor and specialists.
Command(goto=...) handles ALL routing. The supervisor routes to a specialist;
the specialist routes back to the supervisor; the supervisor decides next.
This is exactly what langgraph-supervisor's create_supervisor() builds — no magic.
""")


=== Supervisor multi-agent graph (built by hand with Command) ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	researcher -.-> supervisor;
	supervisor -.-> __end__;
	supervisor -.-> researcher;
	supervisor -.-> writer;
	writer -.-> supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


=== Running ===
  [supervisor] → routing to researcher
  [researcher] Research on 'Explain the LangGraph engine': LangGraph uses a BSP/Pregel engine.
  [supervisor] → routing to writer
  [writer] composed report from 1 finding(s)
  [supervisor] → all done, ending

Final report: REPORT: Research on 'Explain the LangGraph engine': LangGraph uses a BSP/Pregel engine.
Agents that ran: ['researcher', 'writer']

Key point: there are NO explicit edges between supervisor and

In [15]:
# Hierarchical multi-agent: a compiled subgraph used AS A NODE in a parent graph.
from typing import TypedDict, Annotated, Literal
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# Shared state schema (parent and subgraph share the 'findings' + 'messages' keys)
class TeamState(TypedDict):
    task: str
    findings: Annotated[list, add]
    final_report: str

# === Build a "research team" as its OWN graph ===
def junior_researcher(state: TeamState) -> dict:
    print("    [research-team/junior] gathering raw facts")
    return {"findings": ["raw fact: LangGraph = BSP engine"]}

def senior_researcher(state: TeamState) -> dict:
    print("    [research-team/senior] verifying facts")
    return {"findings": ["verified: Pregel runs supersteps"]}

research_builder = StateGraph(TeamState)
research_builder.add_node("junior", junior_researcher)
research_builder.add_node("senior", senior_researcher)
research_builder.add_edge(START, "junior")
research_builder.add_edge("junior", "senior")
research_builder.add_edge("senior", END)
research_team = research_builder.compile()    # ← a compiled graph

# === The TOP-LEVEL graph uses the research_team graph AS A NODE ===
def top_supervisor(state: TeamState) -> Command[Literal["research_team", "writer", "__end__"]]:
    if not state.get("findings"):
        print("  [top] → dispatching the research TEAM (a whole subgraph)")
        return Command(goto="research_team")
    if not state.get("final_report"):
        print("  [top] → dispatching the writer")
        return Command(goto="writer")
    return Command(goto=END)

def writer(state: TeamState) -> Command[Literal["top_supervisor"]]:
    report = f"REPORT from {len(state['findings'])} findings: {'; '.join(state['findings'])}"
    print(f"  [top/writer] composed report")
    return Command(update={"final_report": report}, goto="top_supervisor")

top_builder = StateGraph(TeamState)
top_builder.add_node("top_supervisor", top_supervisor)
top_builder.add_node("research_team", research_team)   # ← THE SUBGRAPH AS A NODE
top_builder.add_node("writer", writer)
top_builder.add_edge(START, "top_supervisor")
top_builder.add_edge("research_team", "top_supervisor")  # team returns to supervisor
top_graph = top_builder.compile()

print("=== Hierarchical graph: a team-subgraph nested inside the top graph ===")
print(top_graph.get_graph().draw_mermaid())

print("\n=== Running (watch the subgraph run its own internal nodes) ===")
result = top_graph.invoke({"task": "Explain LangGraph", "findings": [], "final_report": ""})
print(f"\nFinal report: {result['final_report']}")
print(f"""
Key point: 'research_team' is an ENTIRE COMPILED GRAPH used as a single node.
When the top supervisor routes to it, the whole subgraph runs (junior → senior),
then control returns to the parent. Shared state keys ('findings') flow through
automatically because both graphs use the same schema. This composes infinitely —
teams of teams of teams. That's hierarchical multi-agent in LangGraph.
""")


=== Hierarchical graph: a team-subgraph nested inside the top graph ===
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	top_supervisor(top_supervisor)
	research_team(research_team)
	writer(writer)
	__end__([<p>__end__</p>]):::last
	__start__ --> top_supervisor;
	research_team --> top_supervisor;
	top_supervisor -.-> __end__;
	top_supervisor -.-> research_team;
	top_supervisor -.-> writer;
	writer -.-> top_supervisor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


=== Running (watch the subgraph run its own internal nodes) ===
  [top] → dispatching the research TEAM (a whole subgraph)
    [research-team/junior] gathering raw facts
    [research-team/senior] verifying facts
  [top] → dispatching the writer
  [top/writer] composed report

Final report: REPORT from 2 findings: raw fact: LangGraph = BSP engine; verified: Pregel runs supersteps

Key point: 'research_team' is an ENTIR

---
# Part G: Evaluation & the Ecosystem

## §17 — Evaluation in LangGraph: LangSmith, datasets, trajectory eval

The agentic masterclass N3 covered eval theory deeply. This section is specifically **how you wire eval to a LangGraph agent** — the LangGraph/LangSmith-specific mechanics.

### The setup: LangSmith is LangChain's native eval + tracing platform

Because LangGraph is a LangChain product, LangSmith integration is essentially free — set two environment variables and every graph run is traced automatically:

```python
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "ls-..."
# That's it. Now graph.invoke(...) produces a full trace in LangSmith:
# every node, every LLM call, every tool call, with timing + tokens.
```

No manual instrumentation — the Pregel runtime emits a span per node automatically. This is the agent-aware tracing from N3 §10, with zero code.

### The three eval layers in LangGraph terms

Recall N3 §1: unit / regression / production. Here's how each is built:

#### 1. Unit evals — test discrete graph behavior

Test a single node or a single decision in isolation, like a unit test:

```python
def test_router_picks_tools_on_tool_call():
    state = {"messages": [AIMessage(content="", tool_calls=[{"name": "search", ...}])]}
    assert should_continue(state) == "tools"   # the router function from §9

def test_grade_docs_flags_irrelevant():
    state = {"documents": []}
    result = grade_docs(state)
    assert result["relevant"] is False
```

These are just pytest tests on your node/router functions. **Because nodes are plain functions, they're trivially unit-testable** — pass a state dict, assert on the returned updates. This is a real advantage of the graph model: every piece is independently testable.

#### 2. Regression suite — LLM-as-judge over trajectories on a dataset

Build a LangSmith dataset of (input, reference) pairs, run the full graph on each, judge the output/trajectory:

```python
from langsmith import Client
from agentevals.trajectory.llm import create_trajectory_llm_as_judge

client = Client()
# A dataset of test cases
dataset = client.create_dataset("agent-regression")
client.create_examples(dataset_id=dataset.id, examples=[
    {"inputs": {"question": "weather in Delhi"},
     "outputs": {"reference_answer": "38C clear"}},
    # ... more cases
])

trajectory_judge = create_trajectory_llm_as_judge(model="openai:gpt-5-mini")

def run_agent(inputs):
    return graph.invoke({"messages": [HumanMessage(inputs["question"])]})

# LangSmith runs the agent on every example and applies evaluators
client.evaluate(
    run_agent,
    data="agent-regression",
    evaluators=[trajectory_judge],
    experiment_prefix="v2-prompt-change",
)
```

This runs the whole graph on each test case and scores trajectories with the `agentevals` library from N3 §5. Results land in LangSmith as a comparable experiment.

#### 3. Production sampling — online evals on live traces

LangSmith's online evaluators run on a sampled % of production traces (N3 §10-11). You configure them in the LangSmith UI — pick a sample rate, attach an evaluator, get continuous quality signals on real traffic. No code in your graph.

### What's uniquely testable in LangGraph

The graph model makes certain things easy to evaluate that are hard elsewhere:

| What | How LangGraph makes it easy |
|---|---|
| **Did the right node run?** | Trace shows every node execution |
| **Did the router decide correctly?** | Router is a pure function — unit test it |
| **Trajectory (which path through the graph)** | The sequence of nodes is in the trace |
| **Tool-call correctness** | Tool nodes are traced with args + results |
| **Did the loop terminate correctly?** | Step count + recursion limit visible in trace |
| **State at each step** | Checkpointer history = every intermediate state |

That last one is powerful: because the checkpointer stores every superstep's state (§13 time travel), you can replay exactly what the agent saw at each decision point during debugging.

### The pytest + LangSmith integration

LangSmith plugs into pytest, so eval runs in CI:

```python
import pytest
from langsmith import testing as t

@pytest.mark.langsmith
def test_agent_books_correct_flight():
    result = run_agent({"question": "cheapest flight BLR to BOM tomorrow"})
    t.log_outputs(result)
    score = trajectory_judge(outputs=result["messages"], reference_outputs=...)
    assert score["score"] > 0.8
```

Set thresholds, fail the pipeline when scores drop — bringing the same rigor as deterministic unit tests to agent quality. This is how you gate prompt changes (N3 §15) on eval.

> **Interview note.** *"How do you evaluate a LangGraph agent?"* Three layers. Unit tests on node/router functions (they're plain functions — pass a state, assert on updates). Regression suite via LangSmith datasets + `agentevals` trajectory judges, run on every prompt change. Production sampling via LangSmith online evaluators on a % of live traces. Tracing is automatic — the runtime emits a span per node, no manual instrumentation. The checkpointer's state history makes step-by-step debugging exact.


## §18 — LangGraph vs LlamaIndex vs Google ADK vs OpenAI Agents SDK vs Anthropic SDK

You asked: there must be differences in how these frameworks handle agents, sandboxing, code execution, and architectures — what are they? Here's the real comparison, grounded in the 2026 state of each.

### The convergence, then the divergence

First, the honest truth: **these frameworks have largely converged on the same building blocks.** They all let you define tools, give an LLM a reasoning loop, maintain memory, and customize behavior. The differences are in *philosophy, abstraction level, state handling, and ecosystem* — not in what's fundamentally possible.

That said, the differences matter a lot in practice. Let's go framework by framework.

### LangGraph (LangChain)

**Core abstraction**: a directed graph of nodes + edges over shared state, run by the Pregel/BSP engine.

**Philosophy**: low-level, explicit, maximally flexible. You declare the control flow; nothing is hidden.

| Dimension | LangGraph |
|---|---|
| Orchestration model | Directed graph with conditional edges, cycles |
| State | First-class: typed state + reducers + channels |
| Persistence | Built-in checkpointing with time travel; Postgres/Redis |
| Multi-agent | Supervisor, swarm, hierarchical (subgraphs), scatter-gather (Send) |
| Model support | Fully model-agnostic (any LangChain chat model) |
| Code execution / sandbox | Not built in — you add a sandbox tool (e.g., a Python REPL tool, or an MCP code-exec server). DeepAgents adds sandbox backends |
| Learning curve | Medium — you must learn graph concepts, state, reducers |
| Best at | Maximum control, complex/custom topologies, production with observability (LangSmith) |
| Weakness | More boilerplate than opinionated frameworks; you build more yourself |

**When LangGraph wins**: you need custom control flow, durable state, complex multi-agent topologies, or fine-grained customization. It's the most flexible and the most production-proven (LangSmith observability, used by Replit, Klarna, etc.). The cost is verbosity — you write more code than with opinionated frameworks.

### LlamaIndex (with AgentWorkflow / Workflows)

**Core abstraction**: started as a RAG/data framework; agents are built on its **Workflows** (an event-driven step system) and **AgentWorkflow** orchestration.

**Philosophy**: data-first. Retrieval, indexing, and querying are the center of gravity; agents are an extension.

| Dimension | LlamaIndex |
|---|---|
| Orchestration model | Event-driven Workflows; AgentWorkflow for multi-agent |
| State | Workflow context; less central than LangGraph's typed state |
| Persistence | Workflow checkpointing; less mature than LangGraph's |
| Multi-agent | AgentWorkflow supports handoffs; basic orchestration |
| Model support | Model-agnostic |
| Code execution / sandbox | Via tools; not a core focus |
| Learning curve | Low-medium; very low for RAG specifically |
| Best at | RAG, document-grounded agents, knowledge assistants, anything retrieval-heavy |
| Weakness | Agent orchestration is less expressive than LangGraph; planning-heavy agents are not its strength |

**When LlamaIndex wins**: your application is fundamentally about retrieval over documents. Its indexing, chunking, retrieval, and query-engine abstractions are best-in-class. For a knowledge assistant or RAG-heavy product, LlamaIndex does in a few lines what LangGraph makes you build. **A common 2026 pattern: LlamaIndex for retrieval/memory, LangGraph for orchestration.** They compose.

### Google ADK (Agent Development Kit)

**Core abstraction**: a hierarchical agent tree, software-engineering-first, with rich tooling and Google Cloud integration.

**Philosophy**: production-grade, enterprise, Google-ecosystem. Batteries included — CLI, testing harness, deployment to Vertex AI.

| Dimension | Google ADK |
|---|---|
| Orchestration model | Hierarchical agent tree (sub-agents as a first-class tree structure) |
| State | Session state with pluggable backends |
| Persistence | Session services; integrates with Google Cloud |
| Multi-agent | Native and central — hierarchical agents are the core design |
| Model support | Optimized for Gemini; model-agnostic via LiteLLM (100+ models) |
| Code execution / sandbox | Strong — built-in code executors, including sandboxed execution; Vertex AI integration |
| Multimodal | Best-in-class — native bidirectional audio/video streaming |
| Learning curve | Medium — Google Cloud ecosystem knowledge helps |
| Best at | Multi-agent on Google Cloud, multimodal (voice/video) agents, enterprise with Vertex AI |
| Weakness | Strongest when in the Google ecosystem; less neutral than LangGraph |

**When ADK wins**: you're building multimodal agents (voice, video) or you're on Google Cloud/Vertex AI, or you want a software-engineering-first framework with built-in testing and deployment. Its bidirectional audio/video streaming is genuinely ahead for conversational agents. Launched at Google Cloud NEXT 2025, GA in 2026.


### OpenAI Agents SDK

**Core abstraction**: lightweight agents with **handoffs** as the central primitive. The production successor to the experimental Swarm framework (shipped March 2025/2026).

**Philosophy**: minimal, opinionated, clean. Get an agent running in very little code.

| Dimension | OpenAI Agents SDK |
|---|---|
| Orchestration model | Explicit handoffs — agents transfer control, carrying context |
| State | Context variables (ephemeral by default) |
| Persistence | Not built in by default — you add it |
| Multi-agent | Handoffs are the core primitive; clean and simple |
| Model support | OpenAI models primarily (some flexibility, but OpenAI-first) |
| Code execution / sandbox | Code interpreter tool; sandbox agents in newer versions (0.14+ added sandboxed filesystem tools, Codex-style) |
| Learning curve | Low — clean, opinionated API; minimal concepts |
| Best at | Teams in the OpenAI ecosystem who want first-class model integration with minimal config |
| Weakness | OpenAI-centric; less flexible orchestration than LangGraph; ephemeral state by default |

**When OpenAI Agents SDK wins**: you're committed to OpenAI models and want the simplest path to a working multi-agent system. The handoff primitive is elegant. Trade-off: less control and weaker default persistence than LangGraph, and you're in the OpenAI lane.

### Anthropic Agent SDK (Claude Agent SDK)

**Core abstraction**: a tool-use chain with sub-agents, with **computer use** and **MCP** as first-class primitives. Published alongside Claude 4.x.

**Philosophy**: the "harness" Anthropic uses for Claude Code, exposed. Strong on coding agents, computer use, and MCP-native tooling.

| Dimension | Anthropic Agent SDK |
|---|---|
| Orchestration model | Tool-use chain with sub-agents |
| State | Managed via the conversation + MCP servers |
| Persistence | Via MCP servers / your own storage |
| Multi-agent | Sub-agents (the Claude Code pattern — recall N1 §16 DeepAgents) |
| Model support | Claude models only |
| Code execution / sandbox | Strong — computer use is first-class; bash/file tools; sandboxing via the execution environment |
| Learning curve | Medium — tool-use patterns + MCP understanding |
| Best at | Coding agents, computer-use agents, MCP-native architectures, anything Claude-centric |
| Weakness | Claude models only; less of a general orchestration framework than LangGraph |

**When Anthropic SDK wins**: you're building a coding agent or computer-use agent on Claude, or you want the Claude Code architecture directly. Computer use as a first-class primitive is its standout. It's the closest to "DeepAgents but official" (recall DeepAgents IS the generalized Claude Code architecture).

### The master comparison matrix

| Dimension | LangGraph | LlamaIndex | Google ADK | OpenAI SDK | Anthropic SDK |
|---|---|---|---|---|---|
| **Orchestration** | Directed graph + cycles | Event-driven workflows | Hierarchical agent tree | Explicit handoffs | Tool-use + sub-agents |
| **State persistence** | Built-in checkpointing + time travel | Workflow context | Session state, pluggable | Context vars (ephemeral) | Via MCP / custom |
| **Multi-agent** | Supervisor/swarm/hierarchical/Send | AgentWorkflow handoffs | Hierarchical (native) | Handoffs (native) | Sub-agents |
| **Model support** | Agnostic | Agnostic | Gemini-first, agnostic via LiteLLM | OpenAI-first | Claude only |
| **Code exec / sandbox** | Add-on (tools / DeepAgents backends) | Via tools | Built-in executors, sandboxed | Code interpreter, sandbox agents | Computer use first-class |
| **Multimodal** | Via model | Via model | Best (bidirectional audio/video) | Good | Good (computer use) |
| **Learning curve** | Medium | Low-medium | Medium | Low | Medium |
| **Flexibility** | Highest | Medium | High | Medium | Medium |
| **Best for** | Custom/complex orchestration, production | RAG, knowledge agents | Multimodal, GCP, enterprise | OpenAI-ecosystem simplicity | Coding/computer-use on Claude |

### How they handle sandboxing and code execution (the specific question)

Since you asked specifically about this:

| Framework | Code-execution approach |
|---|---|
| **LangGraph** | No built-in sandbox. You provide a code-execution *tool* (a Python REPL tool, a Docker-backed executor, or an MCP code-exec server). **DeepAgents** (built on LangGraph) adds pluggable sandbox backends — Modal, Daytona, Deno — for isolated execution. So LangGraph itself is unopinionated; the ecosystem fills the gap. |
| **LlamaIndex** | Code execution via tools (e.g., a code interpreter tool). Not a core focus. |
| **Google ADK** | Built-in code executors, including sandboxed execution, with Vertex AI integration. The most "batteries-included" for safe code exec. |
| **OpenAI SDK** | Code interpreter tool; version 0.14+ added sandbox agents with a Codex-style filesystem — sandboxed file operations as a first-class feature. |
| **Anthropic SDK** | Computer use is first-class (the agent controls a real or sandboxed computer); bash/file tools run in the execution environment you provide. The Claude Code lineage means strong file-edit + bash patterns. |

**The pattern**: the cloud-vendor frameworks (ADK, OpenAI, Anthropic) bake in sandboxed execution because they control the runtime. LangGraph and LlamaIndex are runtime-neutral, so they expect *you* to bring the sandbox (via a tool or, for LangGraph, DeepAgents backends). This connects directly to the coding-agent paradigms in the agentic masterclass N2 §7 — the sandbox choice is the defining safety decision.

### The honest recommendation (mix and match)

The mature 2026 take, from practitioners: **don't force one framework to do everything.** A common production stack:

- **LangGraph** for orchestration (the control flow, state, durability).
- **LlamaIndex** for retrieval/memory (its RAG abstractions are best-in-class).
- **MCP** to connect tools across all of them (N2).
- **A2A** if agents span vendor boundaries (N2).

And pick the cloud-vendor SDK (ADK / OpenAI / Anthropic) when you're committed to that ecosystem and want their specific strengths (ADK multimodal, Anthropic computer use, OpenAI simplicity).

> **Interview note.** *"Compare LangGraph to the other agent frameworks."* They've converged on the same primitives (tools, reasoning loop, memory, multi-agent), so differences are philosophy and ecosystem. LangGraph: lowest-level, most flexible, best state/persistence (checkpointing + time travel), model-agnostic, no built-in sandbox. LlamaIndex: RAG-first. Google ADK: hierarchical agents, multimodal, GCP, built-in sandboxed code exec. OpenAI SDK: handoff-centric, minimal, OpenAI-first. Anthropic SDK: tool-use + sub-agents, computer use first-class, Claude-only. The right answer is usually to compose them — LangGraph for orchestration, LlamaIndex for retrieval, MCP/A2A to connect.


## §19 — Maximum customizability and complete clarity

You asked specifically: how to use LangGraph for maximum customizability, and how to understand LangGraph code so well that you have complete flexibility. Here's the distilled answer.

### The clarity principle: there is no hidden control flow

The single most important mental unlock: **in LangGraph, all control flow is explicit.** Unlike frameworks where an opaque "agent executor" decides what happens, in LangGraph *you* declared every node and every edge. When something happens, it's because of an edge you wrote or a `Command`/`Send` a node returned. There is nowhere for magic to hide.

So when you're confused by a LangGraph program, the answer is *always* in one of these places:
1. A node function (what work happened)
2. An edge or conditional edge (why it went there)
3. A `Command`/`Send` return (dynamic routing)
4. A reducer (why state merged that way)
5. The compile config (checkpointer, interrupts)

That's the whole surface area. Master those five and you can read and modify any LangGraph code.

### The customizability ladder (full version)

To customize, climb down only as far as you need:

```
LEVEL 1: create_agent(model, tools)
         → Standard ReAct agent. Zero control flow code.
         → Customize via: system_prompt, tool selection.

LEVEL 2: create_agent(..., middleware=[...])
         → Inject cross-cutting behavior without touching the graph.
         → Customize via: SummarizationMiddleware, HumanInTheLoopMiddleware,
           custom AgentMiddleware (before_model / after_model / wrap_tool_call).

LEVEL 3: StateGraph — build the graph yourself
         → Full control over nodes, edges, state, routing.
         → Customize via: custom nodes, conditional edges, Command, Send, subgraphs.
         → THIS IS WHERE YOU LIVE FOR SERIOUS WORK.

LEVEL 4: Custom reducers, custom state schemas, RetryPolicy/CachePolicy per node
         → Control how state merges, how nodes retry/cache.

LEVEL 5: Functional API (@entrypoint, @task)
         → Imperative style over the engine, when graph-declarative feels wrong.

LEVEL 6: Pregel directly (channels + actors)
         → The bare engine. You almost never need this — but knowing it exists,
           and that everything above is built on it, is what gives total clarity.
```

**The skill is knowing which level a given customization needs.** Most customizations are Level 2 (middleware) or Level 3 (build the graph). You rarely go below Level 4.

### Concrete customization recipes

| You want to... | Level | How |
|---|---|---|
| Add a guardrail before the LLM | 2 | Custom middleware `before_model` hook |
| Summarize long conversations | 2 | `SummarizationMiddleware` |
| Approve sensitive tools | 2 | `HumanInTheLoopMiddleware(interrupt_on=[...])` |
| Insert a node between agent and tools | 3 | Build the graph; add your node, rewire edges |
| Route to different agents by intent | 3 | Conditional edge or `Command` |
| Grade N documents in parallel | 3 | `Send` + a list reducer |
| Retry a flaky API node 3× | 4 | `add_node(..., retry_policy=RetryPolicy(max_attempts=3))` |
| Cache an expensive node | 4 | `add_node(..., cache_policy=CachePolicy(ttl=...))` |
| Keep only last-N messages | 4 | Custom reducer on the messages field |
| Imperative multi-step flow | 5 | Functional API `@entrypoint`/`@task` |

### How to debug any LangGraph agent (the complete method)

1. **Render the graph**: `print(graph.get_graph().draw_mermaid())`. See the structure.
2. **Enable tracing**: set `LANGSMITH_TRACING=true`. See every node, LLM call, tool call.
3. **Stream updates**: `graph.stream(..., stream_mode="updates")`. Watch state evolve node by node.
4. **Inspect state history**: `graph.get_state_history(config)`. See every superstep's state.
5. **Time-travel to the failure**: fork from the checkpoint just before things went wrong.
6. **Unit-test the suspect node/router**: it's a plain function — call it with the state, assert.

Five tools, and you can diagnose anything. No guessing.

### The flexibility payoff

Because LangGraph exposes the engine (not just a high-level agent), you can build things no opinionated framework allows:

- **Arbitrary topologies** — not just ReAct or supervisor; any graph you can draw.
- **Custom state semantics** — reducers let you define exactly how state merges.
- **Mixed sync/async** — per-node, as needed.
- **Deep persistence control** — checkpoint where you want, resume how you want, time-travel.
- **Compose with anything** — a node can call LlamaIndex, an MCP server, another framework's agent, a raw HTTP API.

That last point is the ultimate flexibility: **a LangGraph node is just a Python function, so it can do literally anything.** The graph gives you orchestration, state, and persistence around arbitrary Python. Nothing is off-limits.

### The reading drill (to build total clarity)

To get to "complete clarity," do this with three open-source LangGraph agents:

1. Find the State class → understand the data model.
2. Render with `draw_mermaid()` → see the topology.
3. Read each node function → understand each unit of work.
4. Read each router → understand each decision.
5. Find the compile config → understand persistence + interrupts.
6. Trace one full run mentally → predict the path, then verify with `stream`.

Do this three times and LangGraph holds no mysteries. The framework is small once you see that it's *just* state + nodes + edges + the Pregel engine underneath.

> **Interview note.** *"How do you get maximum flexibility out of LangGraph?"* Work at the StateGraph level (not just `create_agent`), because that exposes nodes, edges, `Command`, `Send`, subgraphs, custom reducers, and per-node retry/cache policies. A node is just a Python function, so it can call anything — other frameworks, MCP servers, raw APIs. For clarity: all control flow is explicit (nodes, edges, Command/Send, reducers, compile config — that's the entire surface area), and you debug with draw_mermaid + tracing + stream + state history + time travel + unit-testing the plain-function nodes.


---
# Synthesis & Cheat Sheet

## The complete LangGraph picture

Everything in this notebook, in one mental model:

```
   ┌─────────────────────────────────────────────────────────────┐
   │  YOU DECLARE:                                                 │
   │                                                               │
   │   State (TypedDict + reducers)  ──┐                           │
   │   Nodes (functions)             ──┼──► StateGraph builder     │
   │   Edges (normal + conditional)  ──┘         │                 │
   │                                             │ .compile()      │
   │                                             ▼                 │
   │                                    CompiledStateGraph         │
   └─────────────────────────────────────────┬───────────────────┘
                                              │
                              runs on ────────┘
                                              ▼
   ┌─────────────────────────────────────────────────────────────┐
   │  THE PREGEL / BSP ENGINE (the runtime):                      │
   │                                                               │
   │   loop of supersteps:                                         │
   │     PLAN    → which nodes have new input?                    │
   │     EXECUTE → run them in parallel (sibling-isolated)        │
   │     UPDATE  → merge writes via reducers (the barrier)        │
   │     CHECKPOINT → persist state to the checkpointer           │
   │   ...until no node has work (reach END)                      │
   └─────────────────────────────────────────────────────────────┘
                              │                    │
                  short-term  │                    │  long-term
                              ▼                    ▼
                    ┌──────────────────┐  ┌──────────────────┐
                    │  CHECKPOINTER     │  │  STORE            │
                    │  per thread_id    │  │  per namespace    │
                    │  → resume, time   │  │  → cross-thread   │
                    │    travel         │  │    memory         │
                    └──────────────────┘  └──────────────────┘
```

## Cheat sheet

### The primitives

| Primitive | What | Code |
|---|---|---|
| State | Shared typed data | `class S(TypedDict): x: Annotated[list, add]` |
| Node | A function (state)→updates | `builder.add_node("n", fn)` |
| Normal edge | Unconditional next | `builder.add_edge("a", "b")` |
| Entry / exit | Start / end | `add_edge(START, "a")` / `add_edge("z", END)` |
| Conditional edge | Router function | `add_conditional_edges("a", router, path_map)` |
| Command | Update + route in one | `return Command(update={...}, goto="b")` |
| Send | Dynamic parallel fan-out | `return [Send("n", {...}) for x in items]` |
| Compile | Make it runnable | `graph = builder.compile(checkpointer=..., store=...)` |

### Reducers

| Field | Reducer |
|---|---|
| One writer, overwrite | (none — default) |
| Messages | `Annotated[list, add_messages]` |
| Growing list | `Annotated[list, operator.add]` |
| Counter | `Annotated[int, operator.add]` |
| Custom merge | `Annotated[T, my_reducer_fn]` |

### The engine (Pregel / BSP)

- Execution = supersteps: **plan → execute → update → checkpoint**.
- Nodes communicate only through channels (state fields).
- Within a superstep, siblings don't see each other's writes (isolation).
- Concurrent writes to one field require a reducer, else `InvalidUpdateError`.
- Nodes triggered together run in parallel automatically.

### Memory

| Type | Mechanism | Scope |
|---|---|---|
| Working | State (channels) | This run |
| Short-term | Checkpointer (`thread_id`) | This conversation |
| Long-term | Store (namespace) | Across conversations |

Checkpointers: `MemorySaver` (dev) → `SqliteSaver` (local) → `PostgresSaver` (prod).

### Agents

- From scratch: agent node + tools node + conditional edge (loop) + `tools→agent` edge.
- Prebuilt: `create_agent(model, tools, system_prompt)`.
- `ToolNode` for the tools node; middleware for cross-cutting behavior.

### Control flow for multi-agent

| Pattern | Mechanism |
|---|---|
| Supervisor | central node + `Command(goto=specialist)` |
| Swarm | handoff tools → `Command(goto=peer, graph=Command.PARENT)` |
| Hierarchical | compiled subgraph as a node |
| Scatter-gather | `Send` + reducer |

### HITL / streaming / time travel

- HITL: `interrupt(payload)` in a node, resume with `Command(resume=value)`. Needs checkpointer.
- Stream modes: `"values"` (full state), `"updates"` (diffs), `"messages"` (tokens), `"custom"`.
- Time travel: `get_state_history()`, fork from a checkpoint, `update_state()` to edit.

### The customizability ladder

```
create_agent → +middleware → StateGraph → custom reducers/policies → Functional API → Pregel
   (simplest)                  (most work lives here)                          (rarely)
```

### Debugging method

1. `draw_mermaid()` — see structure
2. `LANGSMITH_TRACING=true` — trace everything
3. `stream(stream_mode="updates")` — watch it run
4. `get_state_history()` — inspect every step
5. time-travel to the failure point
6. unit-test the suspect node (it's a plain function)

### Framework one-liners

| Framework | One-liner |
|---|---|
| LangGraph | Lowest-level, most flexible graph engine; best state/persistence; model-agnostic |
| LlamaIndex | RAG-first; best retrieval abstractions |
| Google ADK | Hierarchical agents, multimodal, GCP, built-in sandboxed code exec |
| OpenAI SDK | Handoff-centric, minimal, OpenAI-first |
| Anthropic SDK | Tool-use + sub-agents, computer use first-class, Claude-only |

## Interview narration

### 90-second "explain LangGraph"

> "LangGraph models an agent as a directed graph: nodes are functions that do work, edges declare control flow, and all nodes share a typed state object. It runs on a Pregel / Bulk-Synchronous-Parallel engine — execution proceeds in supersteps that plan which nodes to run, execute them in parallel, merge their state writes via reducers at a barrier, then checkpoint. That checkpointing is what gives you durable persistence, resume, and time travel for free. You get the agent loop by wiring a conditional edge from the model node — tool calls route to a tools node, else END — with the tools node looping back to the model. State updates merge via reducers; the canonical `messages` field uses `add_messages` which appends and dedupes by ID. Memory is two systems: the checkpointer (per thread_id, short-term/conversation) and the store (per namespace, long-term/cross-conversation). For multi-agent, agents are nodes or subgraphs routed with `Command` and `Send`. It's the lowest-level, most flexible agent framework, and the most production-proven via LangSmith."

### The mental model to carry

LangGraph is **state + nodes + edges, run by a superstep engine that checkpoints after every step.** Everything else — agents, RAG, multi-agent, memory, HITL, streaming, time travel — is a composition of those primitives. There's no hidden control flow; you declared all of it. That's the source of both its flexibility and its clarity.

## What you can now do

- Explain the LangGraph engine (Pregel/BSP) and why it enables persistence + parallelism.
- Read any LangGraph codebase by finding state, nodes, edges, routers, compile config.
- Build a ReAct agent from scratch and with `create_agent`.
- Build RAG (including corrective RAG) and multi-agent systems (supervisor/swarm/hierarchical/scatter-gather).
- Handle memory correctly (checkpointer vs store), HITL (interrupt/resume), streaming, time travel.
- Evaluate agents (unit / regression / production) with LangSmith + agentevals.
- Choose between LangGraph, LlamaIndex, ADK, OpenAI SDK, Anthropic SDK — and compose them.
- Customize at the right level and debug anything.

You wanted complete clarity and complete flexibility with LangGraph. You now have the model for both: it's a small framework once you see the engine underneath. The rest is reps — build three real graphs and it's yours.

---

*End of the LangChain & LangGraph Masterclass.*
